# Notebook_01_Furusato_Prepare_Ontology_Data

**Workshop version:** 2.7.0  
**Dataset contract:** 2.7.0-realistic.1 (`generated-realistic-v1`)

This Fabric PySpark notebook validates the eight canonical seed CSV files before
creating typed staging data and the audited ontology-ready model. Static Donation
rows are a synthetic snapshot; they are not realtime observations or lifecycle
status records.

Publication uses validated, run-scoped Delta tables and then replaces each final
table in a controlled sequence. Each Delta table update is transactional by itself,
but Fabric does not provide a cross-table ACID transaction for this sequence.
On publication failure the notebook attempts per-table Delta restore and retains
the validated temporary tables for diagnosis.



In [ ]:
# Fabric parameter cell
NOTEBOOK_VERSION = "2.7.0"
PARTICIPANT_ID = "001"
INPUT_DIR = "Files/furusato/seed"
RUN_OPTIMIZE_VORDER = False
PUBLISH_LEASE_MINUTES = 30
OPERATOR_RECOVER_STALE_LEASE = False
STALE_LEASE_OWNER_RUN_ID = ""
STALE_LEASE_GENERATION = -1


## Canonical contracts

The checksum values below come from the generated `dataset-manifest.json`, not from
unavailable historical CSV bytes. Ranks are deterministic one-based ordinals:
descending amount followed by the stable entity ID as the tie-breaker.



In [ ]:
from datetime import datetime, timedelta, timezone
from functools import reduce
from pathlib import Path
import hashlib
import json
import os
import re
import uuid

from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")

DATASET_VERSION = "2.7.0-realistic.1"
MANIFEST_VERSION = 1
CHECKSUM_CONTRACT = "generated-realistic-v1"
DONATION_DATA_LAYER = "StaticSyntheticSnapshot"
PUBLISH_CONTROL_TABLE = "audit_furusato_publish_control"
PUBLISH_CONTROL_KEY = "furusato-canonical-v2.7.0"
PUBLISH_CONTROL_RECORD = "CONTROL"
WORKSHOP_GENERATION_COLUMN = "_WorkshopGenerationId"

INPUT_CONTRACT = {
    "prefectures.csv": {
        "rows": 47,
        "header": "PrefectureID,PrefectureName,PrefectureNameEn",
        "sha256": "0b534b8c5e9aacf659ef52599487096c9df9d6e9324ad659b9dcb0e36f1846cc",
        "stage": "stg_prefectures",
    },
    "municipalities.csv": {
        "rows": 1741,
        "header": "MunicipalityID,MunicipalityName,PrefectureID,PrefectureName",
        "sha256": "8c344cbd7663b8c496e68a3cbe92136b3e5789da65d3795858f42f63b4d9ac63",
        "stage": "stg_municipalities",
    },
    "donors.csv": {
        "rows": 12000,
        "header": "DonorID,DonorName,PrefectureID,Age,Occupation,PrefectureName,PrefectureNameEn,OccupationEn",
        "sha256": "e5772173373ffb19c5203d6ecb26329327c002adc83292ada432341ff0cf4fb3",
        "stage": "stg_donors",
    },
    "categories.csv": {
        "rows": 30,
        "header": "CategoryID,CategoryName,CategoryNameEn",
        "sha256": "78bc570c40430d6b1970e8997adeec81e6442fda31601603efc69bbe687ebc26",
        "stage": "stg_categories",
    },
    "gifts.csv": {
        "rows": 6000,
        "header": "GiftID,GiftName,CategoryID,MunicipalityID,Notes,GiftNameEn",
        "sha256": "49650ad859da5a6765a962cbf5f68bd4238703bb0f61864fabe06a0f06a71869",
        "stage": "stg_gifts",
    },
    "businesses.csv": {
        "rows": 2500,
        "header": "BusinessID,BusinessName,PrefectureID,BusinessType,PrefectureName,BusinessTypeEn",
        "sha256": "41534c77287abe881ab33403ac10078097118c64ce694351fe24605f399f7c97",
        "stage": "stg_businesses",
    },
    "business_gifts.csv": {
        "rows": 14514,
        "header": "BusinessID,GiftID",
        "sha256": "63b359660228e4166598a7779a83b3246c47cd4d202b69f5efbf5ab54c6a8464",
        "stage": "stg_business_gifts",
    },
    "donation_orders.csv": {
        "rows": 80000,
        "header": "DonationID,DonorID,MunicipalityID,GiftID,DonationAmountYen,DonatedAt,PaymentMethod",
        "sha256": "2995bf66896993614090e895ef1814b0383122bd837dba4b948b30e8a57cfadf",
        "stage": "stg_donation_orders",
    },
}

STAGING_SCHEMA_CONTRACT = {
    "stg_prefectures": [
        ("PrefectureID", "bigint"), ("PrefectureName", "string"),
        ("PrefectureNameEn", "string"),
    ],
    "stg_municipalities": [
        ("MunicipalityID", "string"), ("MunicipalityName", "string"),
        ("PrefectureID", "bigint"), ("PrefectureName", "string"),
    ],
    "stg_donors": [
        ("DonorID", "bigint"), ("DonorName", "string"),
        ("PrefectureID", "bigint"), ("Age", "bigint"),
        ("Occupation", "string"), ("PrefectureName", "string"),
        ("PrefectureNameEn", "string"), ("OccupationEn", "string"),
    ],
    "stg_categories": [
        ("CategoryID", "bigint"), ("CategoryName", "string"),
        ("CategoryNameEn", "string"),
    ],
    "stg_gifts": [
        ("GiftID", "bigint"), ("GiftName", "string"),
        ("CategoryID", "bigint"), ("MunicipalityID", "string"),
        ("Notes", "string"), ("GiftNameEn", "string"),
    ],
    "stg_businesses": [
        ("BusinessID", "bigint"), ("BusinessName", "string"),
        ("PrefectureID", "bigint"), ("BusinessType", "string"),
        ("PrefectureName", "string"), ("BusinessTypeEn", "string"),
    ],
    "stg_business_gifts": [
        ("BusinessID", "bigint"), ("GiftID", "bigint"),
    ],
    "stg_donation_orders": [
        ("DonationID", "bigint"), ("DonorID", "bigint"),
        ("MunicipalityID", "string"), ("GiftID", "bigint"),
        ("DonationAmountYen", "bigint"), ("DonatedAt", "timestamp"),
        ("PaymentMethod", "string"),
    ],
}

STATIC_PROPERTY_CONTRACT = {
    "ot_prefecture": [
        ("PrefectureId", "bigint"),
        ("PrefectureName", "string"),
        ("PrefectureNameEn", "string"),
        ("PrefReceivedStaticCount", "bigint"),
        ("PrefReceivedTotalYen", "bigint"),
        ("PrefReceivedAmountRank", "bigint"),
        ("PrefResidentStaticCount", "bigint"),
        ("PrefResidentTotalYen", "bigint"),
        ("PrefResidentAmountRank", "bigint"),
    ],
    "ot_municipality": [
        ("MunicipalityId", "string"),
        ("MunicipalityName", "string"),
        ("MunicipalityDisplayName", "string"),
        ("MunicipalityStaticCount", "bigint"),
        ("MunicipalityStaticTotalYen", "bigint"),
        ("MunicipalityAmountRank", "bigint"),
        ("MunicipalityPrefAmountRank", "bigint"),
    ],
    "ot_donor": [
        ("DonorId", "bigint"),
        ("DonorName", "string"),
        ("DonorDisplayName", "string"),
        ("DonorAge", "bigint"),
        ("DonorOccupation", "string"),
        ("DonorOccupationEn", "string"),
        ("DonorStaticCount", "bigint"),
        ("DonorStaticTotalYen", "bigint"),
        ("DonorStaticMaxYen", "bigint"),
        ("DonorOverallAmountRank", "bigint"),
        ("DonorResidenceAmountRank", "bigint"),
    ],
    "ot_gift_category": [
        ("CategoryId", "bigint"),
        ("CategoryName", "string"),
        ("CategoryNameEn", "string"),
        ("CategorySearchTerms", "string"),
        ("CategoryStaticCount", "bigint"),
        ("CategoryStaticTotalYen", "bigint"),
        ("CategoryAmountRank", "bigint"),
    ],
    "ot_gift": [
        ("GiftId", "bigint"),
        ("GiftName", "string"),
        ("GiftDisplayName", "string"),
        ("GiftSearchTerms", "string"),
        ("GiftNotes", "string"),
        ("GiftStaticCount", "bigint"),
        ("GiftStaticTotalYen", "bigint"),
        ("GiftAmountRank", "bigint"),
    ],
    "ot_supplier": [
        ("SupplierId", "bigint"),
        ("SupplierName", "string"),
        ("SupplierDisplayName", "string"),
        ("SupplierType", "string"),
        ("SupplierTypeEn", "string"),
        ("SupplierProvidedGiftCount", "bigint"),
    ],
    "ot_donation": [
        ("DonationId", "bigint"),
        ("DonationDisplayName", "string"),
        ("DonationAmountYen", "bigint"),
        ("DonatedAtUtc", "timestamp"),
        ("DonatedAtJstText", "string"),
        ("DonationDateJst", "string"),
        ("DonationYearMonthJst", "string"),
        ("DonationYearMonthJaShort", "string"),
        ("DonationPaymentMethod", "string"),
        ("DonationPaymentMethodEn", "string"),
        ("DonationDataLayer", "string"),
    ],
    "ot_mun_category_metric": [
        ("MunCategoryMetricId", "string"),
        ("MunCategoryStaticCount", "bigint"),
        ("MunCategoryTotalYen", "bigint"),
        ("MunCategoryAmountRank", "bigint"),
    ],
    "ot_pref_category_metric": [
        ("PrefCategoryMetricId", "string"),
        ("PrefCategoryStaticCount", "bigint"),
        ("PrefCategoryTotalYen", "bigint"),
        ("PrefCategoryAmountRank", "bigint"),
    ],
    "ot_pref_donation_flow": [
        ("PrefDonationFlowId", "string"),
        ("PrefFlowStaticCount", "bigint"),
        ("PrefFlowTotalYen", "bigint"),
        ("PrefFlowOriginRank", "bigint"),
        ("PrefFlowDestinationRank", "bigint"),
    ],
}

RELATION_COLUMN_CONTRACT = {
    "ot_prefecture": [],
    "ot_municipality": [("PrefectureId", "bigint")],
    "ot_donor": [("PrefectureId", "bigint")],
    "ot_gift_category": [],
    "ot_gift": [("CategoryId", "bigint"), ("MunicipalityId", "string")],
    "ot_supplier": [("PrefectureId", "bigint")],
    "ot_supplier_gift": [("SupplierId", "bigint"), ("GiftId", "bigint")],
    "ot_donation": [
        ("DonorId", "bigint"), ("MunicipalityId", "string"),
        ("GiftId", "bigint"),
    ],
    "ot_mun_category_metric": [
        ("MunicipalityId", "string"), ("CategoryId", "bigint"),
    ],
    "ot_pref_category_metric": [
        ("PrefectureId", "bigint"), ("CategoryId", "bigint"),
    ],
    "ot_pref_donation_flow": [
        ("ResidencePrefectureId", "bigint"),
        ("RecipientPrefectureId", "bigint"),
    ],
}

OUTPUT_ROW_COUNTS = {
    "ot_prefecture": 47,
    "ot_municipality": 1741,
    "ot_donor": 12000,
    "ot_gift_category": 30,
    "ot_gift": 6000,
    "ot_supplier": 2500,
    "ot_supplier_gift": 14514,
    "ot_donation": 80000,
    "ot_mun_category_metric": 4403,
    "ot_pref_category_metric": 662,
    "ot_pref_donation_flow": 2209,
}

OUTPUT_SCHEMA_CONTRACT = {
    name: STATIC_PROPERTY_CONTRACT.get(name, []) + RELATION_COLUMN_CONTRACT[name]
    for name in RELATION_COLUMN_CONTRACT
}
STAGE_TABLE_ORDER = [item["stage"] for item in INPUT_CONTRACT.values()]
OUTPUT_TABLE_ORDER = list(OUTPUT_ROW_COUNTS)
AUDIT_TABLE = "audit_furusato_load_manifest"
FINAL_TABLE_ORDER = STAGE_TABLE_ORDER + OUTPUT_TABLE_ORDER + [AUDIT_TABLE]

static_property_names = [
    name
    for properties in STATIC_PROPERTY_CONTRACT.values()
    for name, _ in properties
]
assert len(static_property_names) == 72
assert len(static_property_names) == len(set(static_property_names))
assert all(1 <= len(name) <= 26 for name in static_property_names)

assert isinstance(PARTICIPANT_ID, str) and re.fullmatch(
    r"(?!000$)[0-9]{3}", PARTICIPANT_ID
), 'PARTICIPANT_ID must be a value from "001" through "999".'
assert isinstance(RUN_OPTIMIZE_VORDER, bool), (
    "RUN_OPTIMIZE_VORDER must be True or False."
)
assert isinstance(PUBLISH_LEASE_MINUTES, int) and 15 <= PUBLISH_LEASE_MINUTES <= 1440, (
    "PUBLISH_LEASE_MINUTES must be an int between 15 and 1440. The lease is renewed "
    "before every table, so a shorter value only shortens the wait before a crashed "
    "run can be recovered."
)
assert isinstance(OPERATOR_RECOVER_STALE_LEASE, bool), (
    "OPERATOR_RECOVER_STALE_LEASE must be True or False."
)
assert isinstance(STALE_LEASE_OWNER_RUN_ID, str), (
    "STALE_LEASE_OWNER_RUN_ID must be a string."
)
assert isinstance(STALE_LEASE_GENERATION, int), (
    "STALE_LEASE_GENERATION must be an int."
)
if OPERATOR_RECOVER_STALE_LEASE:
    assert STALE_LEASE_OWNER_RUN_ID, (
        "STALE_LEASE_OWNER_RUN_ID is required. Copy the OwnerRunId value printed by "
        "the stale publication lease error."
    )
    assert STALE_LEASE_GENERATION >= 1, (
        "STALE_LEASE_GENERATION is required. Copy the Generation value printed by "
        "the stale publication lease error."
    )
assert isinstance(INPUT_DIR, str) and re.fullmatch(
    r"Files/[A-Za-z0-9._/-]+", INPUT_DIR
), 'INPUT_DIR must be a relative Lakehouse path, for example "Files/furusato/seed".'
assert ".." not in INPUT_DIR.split("/"), (
    "INPUT_DIR must not contain a parent-directory segment."
)

RUN_ID = f"p{PARTICIPANT_ID}-{uuid.uuid4().hex[:16]}"


## Validation helpers

Every input condition is checked before any Delta write. The same schema, key,
relationship, row-count, reconciliation, and accuracy checks are repeated against
materialized temporary tables and again after controlled publication.



In [ ]:
TABLE_NAME_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def require(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)


def quoted_table(name: str) -> str:
    require(bool(TABLE_NAME_PATTERN.fullmatch(name)), f"Unsafe table name: {name}")
    return f"`{name}`"


def has_rows(df: DataFrame, condition) -> bool:
    return df.where(condition).limit(1).count() > 0


def assert_row_count(df: DataFrame, expected: int, label: str) -> None:
    actual = df.count()
    require(actual == expected, f"{label}: expected {expected} rows, found {actual}.")


def assert_no_nulls(df: DataFrame, columns: list[str], label: str) -> None:
    conditions = [F.col(column).isNull() for column in columns]
    if conditions:
        require(
            not has_rows(df, reduce(lambda left, right: left | right, conditions)),
            f"{label}: null value found in required columns.",
        )


def assert_no_blank_strings(df: DataFrame, columns: list[str], label: str) -> None:
    conditions = [
        F.col(column).isNull() | (F.length(F.col(column)) == 0)
        for column in columns
    ]
    if conditions:
        require(
            not has_rows(df, reduce(lambda left, right: left | right, conditions)),
            f"{label}: blank string found in required columns.",
        )


def assert_regex(df: DataFrame, column: str, pattern: str, label: str) -> None:
    require(
        not has_rows(df, ~F.col(column).rlike(pattern)),
        f"{label}: {column} violates {pattern}.",
    )


def assert_unique(df: DataFrame, columns: list[str], label: str) -> None:
    duplicate = (
        df.groupBy(*columns)
        .count()
        .where(F.col("count") != 1)
        .limit(1)
        .count()
    )
    require(duplicate == 0, f"{label}: key is not unique: {columns}.")


def assert_fk(
    child: DataFrame,
    child_columns: list[str],
    parent: DataFrame,
    parent_columns: list[str],
    label: str,
) -> None:
    require(len(child_columns) == len(parent_columns), f"{label}: invalid FK shape.")
    aliases = [f"__fk_{index}" for index in range(len(child_columns))]
    left = child.select(
        *[F.col(column).alias(alias) for column, alias in zip(child_columns, aliases)]
    ).distinct()
    right = parent.select(
        *[F.col(column).alias(alias) for column, alias in zip(parent_columns, aliases)]
    ).distinct()
    missing = left.join(right, aliases, "left_anti").limit(1).count()
    require(missing == 0, f"{label}: orphan foreign key found.")


def assert_schema(
    df: DataFrame, expected: list[tuple[str, str]], label: str
) -> None:
    actual = [
        (field.name, field.dataType.simpleString())
        for field in df.schema.fields
    ]
    require(actual == expected, f"{label}: schema mismatch: {actual}.")


def assert_rank_matches_order(
    df: DataFrame,
    rank_column: str,
    metric_column: str,
    stable_id_columns: list[str],
    partition_columns: list[str],
    label: str,
) -> None:
    ordering = [F.desc(metric_column)] + [
        F.asc(column) for column in stable_id_columns
    ]
    ranking_window = (
        Window.partitionBy(*partition_columns).orderBy(*ordering)
        if partition_columns
        else Window.orderBy(*ordering)
    )
    expected = df.withColumn(
        "__ExpectedDeterministicRank",
        F.row_number().over(ranking_window).cast("long"),
    )
    mismatched = expected.where(
        F.col(rank_column) != F.col("__ExpectedDeterministicRank")
    ).limit(1).count()
    require(
        mismatched == 0,
        f"{label}: rank does not match descending {metric_column} and stable-ID ordering.",
    )


def literal_map(values: dict[str, str]):
    expressions = []
    for key, value in values.items():
        expressions.extend([F.lit(key), F.lit(value)])
    return F.create_map(*expressions)


def lakehouse_local_path(relative_path: str) -> Path:
    require(relative_path.startswith("Files/"), "Input must use a Lakehouse Files path.")
    require(".." not in relative_path.split("/"), "Parent path segments are not allowed.")
    return Path("/lakehouse/default") / relative_path


def sha256_bytes(content: bytes) -> str:
    return hashlib.sha256(content).hexdigest()


def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(name)


def drop_tables_best_effort(names: list[str]) -> list[str]:
    errors = []
    for name in names:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {quoted_table(name)}")
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    return errors


def long_text_is_valid(df: DataFrame, column: str, minimum: int = 0) -> bool:
    casted = F.col(column).cast("long")
    return not has_rows(
        df,
        (~F.col(column).rlike(r"^[0-9]+$"))
        | casted.isNull()
        | (casted < F.lit(minimum)),
    )



## Inventory, UTF-8, header, checksum, and source row validation

Only the eight manifest-listed filenames are accepted. Byte validation rejects a
BOM, malformed UTF-8, CR/CRLF data, a noncanonical header, a missing final LF, and a
checksum mismatch before Spark reads any record.



In [ ]:
expected_file_names = sorted(INPUT_CONTRACT)
input_entries = notebookutils.fs.ls(INPUT_DIR)
actual_file_names = sorted(
    os.path.basename(entry.path.rstrip("/")) for entry in input_entries
)
require(
    actual_file_names == expected_file_names,
    f"Seed filename mismatch. Expected={expected_file_names} Actual={actual_file_names}",
)

local_input_dir = lakehouse_local_path(INPUT_DIR)
actual_hashes = {}
byte_sizes = {}
for file_name, contract in INPUT_CONTRACT.items():
    file_path = local_input_dir / file_name
    require(file_path.is_file(), f"Default Lakehouse mount cannot read {file_path}.")
    content = file_path.read_bytes()
    require(not content.startswith(b"\xef\xbb\xbf"), f"{file_name}: UTF-8 BOM is not allowed.")
    try:
        text = content.decode("utf-8", errors="strict")
    except UnicodeDecodeError as exc:
        raise AssertionError(f"{file_name}: invalid UTF-8: {exc}") from exc
    require("\r" not in text, f"{file_name}: generated canonical data must use LF.")
    require(text.endswith("\n"), f"{file_name}: generated canonical data must end in LF.")
    actual_header = text.split("\n", 1)[0]
    require(
        actual_header == contract["header"],
        f"{file_name}: header mismatch: {actual_header}",
    )
    actual_hash = sha256_bytes(content)
    require(
        actual_hash == contract["sha256"],
        f"{file_name}: SHA-256 mismatch. Expected={contract['sha256']} Actual={actual_hash}",
    )
    actual_hashes[file_name] = actual_hash
    byte_sizes[file_name] = len(content)


def read_all_strings(file_name: str) -> DataFrame:
    columns = INPUT_CONTRACT[file_name]["header"].split(",")
    schema = T.StructType(
        [T.StructField(column, T.StringType(), True) for column in columns]
    )
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("mode", "FAILFAST")
        .option("quote", '"')
        .option("escape", '"')
        .schema(schema)
        .load(f"{INPUT_DIR}/{file_name}")
    )


raw = {file_name: read_all_strings(file_name) for file_name in INPUT_CONTRACT}
for file_name, df in raw.items():
    contract = INPUT_CONTRACT[file_name]
    expected_columns = contract["header"].split(",")
    require(df.columns == expected_columns, f"{file_name}: Spark column mismatch.")
    assert_row_count(df, contract["rows"], file_name)
    assert_no_blank_strings(df, expected_columns, file_name)

integer_columns = {
    "prefectures.csv": ["PrefectureID"],
    "municipalities.csv": ["PrefectureID"],
    "donors.csv": ["DonorID", "PrefectureID", "Age"],
    "categories.csv": ["CategoryID"],
    "gifts.csv": ["GiftID", "CategoryID"],
    "businesses.csv": ["BusinessID", "PrefectureID"],
    "business_gifts.csv": ["BusinessID", "GiftID"],
    "donation_orders.csv": [
        "DonationID", "DonorID", "GiftID", "DonationAmountYen",
    ],
}
for file_name, columns in integer_columns.items():
    for column in columns:
        require(
            long_text_is_valid(raw[file_name], column, 1),
            f"{file_name}: {column} is not a positive 64-bit integer.",
        )

for file_name in ["municipalities.csv", "gifts.csv", "donation_orders.csv"]:
    assert_regex(raw[file_name], "MunicipalityID", r"^[0-9]{6}$", file_name)

require(
    not has_rows(
        raw["prefectures.csv"],
        ~F.col("PrefectureID").cast("long").between(1, 47),
    ),
    "prefectures.csv: PrefectureID is outside 1..47.",
)
require(
    not has_rows(
        raw["donors.csv"],
        ~F.col("Age").cast("long").between(0, 120),
    ),
    "donors.csv: Age is outside 0..120.",
)
require(
    not has_rows(
        raw["donation_orders.csv"],
        ~F.col("DonatedAt").rlike(
            r"^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}Z$"
        )
        | F.to_timestamp(
            "DonatedAt", "yyyy-MM-dd'T'HH:mm:ss'Z'"
        ).isNull(),
    ),
    "donation_orders.csv: DonatedAt is not a canonical UTC timestamp.",
)
require(
    not has_rows(
        raw["donation_orders.csv"],
        ~F.col("PaymentMethod").isin(
            "クレジットカード", "銀行振込", "コンビニ決済", "電子決済"
        ),
    ),
    "donation_orders.csv: unsupported PaymentMethod.",
)
require(
    not has_rows(
        raw["businesses.csv"],
        ~F.col("BusinessType").isin(
            "スポーツ用品",
            "ペット用品",
            "伝統工芸",
            "化粧品製造",
            "園芸・造園",
            "文具・玩具",
            "木工・家具",
            "果樹・青果",
            "水産・水産加工",
            "生活用品",
            "畜産・食肉加工",
            "窯業・陶磁器",
            "米穀・製粉",
            "総合物産",
            "繊維・アパレル",
            "製茶・焙煎",
            "製菓・製パン",
            "製麺",
            "観光・体験",
            "観光・宿泊",
            "調味料製造",
            "農産・青果",
            "酒造",
            "酪農・乳製品",
            "醸造",
            "電機・機械",
            "食品加工",
            "飲料製造",
        ),
    ),
    "businesses.csv: unsupported BusinessType.",
)



## Typed staging frames and relational validation

Source headers are retained in staging tables. Numeric identifiers and amounts are
64-bit integers, `MunicipalityID` remains a six-character string, and `DonatedAt`
becomes a UTC Spark timestamp. All PK, composite-key, FK, name-consistency, and
donation-to-gift municipality conditions are validated before a Delta write.



In [ ]:
stages = {
    "stg_prefectures": raw["prefectures.csv"].select(
        F.col("PrefectureID").cast("long").alias("PrefectureID"),
        "PrefectureName",
        "PrefectureNameEn",
    ),
    "stg_municipalities": raw["municipalities.csv"].select(
        "MunicipalityID",
        "MunicipalityName",
        F.col("PrefectureID").cast("long").alias("PrefectureID"),
        "PrefectureName",
    ),
    "stg_donors": raw["donors.csv"].select(
        F.col("DonorID").cast("long").alias("DonorID"),
        "DonorName",
        F.col("PrefectureID").cast("long").alias("PrefectureID"),
        F.col("Age").cast("long").alias("Age"),
        "Occupation",
        "PrefectureName",
        "PrefectureNameEn",
        "OccupationEn",
    ),
    "stg_categories": raw["categories.csv"].select(
        F.col("CategoryID").cast("long").alias("CategoryID"),
        "CategoryName",
        "CategoryNameEn",
    ),
    "stg_gifts": raw["gifts.csv"].select(
        F.col("GiftID").cast("long").alias("GiftID"),
        "GiftName",
        F.col("CategoryID").cast("long").alias("CategoryID"),
        "MunicipalityID",
        "Notes",
        "GiftNameEn",
    ),
    "stg_businesses": raw["businesses.csv"].select(
        F.col("BusinessID").cast("long").alias("BusinessID"),
        "BusinessName",
        F.col("PrefectureID").cast("long").alias("PrefectureID"),
        "BusinessType",
        "PrefectureName",
        "BusinessTypeEn",
    ),
    "stg_business_gifts": raw["business_gifts.csv"].select(
        F.col("BusinessID").cast("long").alias("BusinessID"),
        F.col("GiftID").cast("long").alias("GiftID"),
    ),
    "stg_donation_orders": raw["donation_orders.csv"].select(
        F.col("DonationID").cast("long").alias("DonationID"),
        F.col("DonorID").cast("long").alias("DonorID"),
        "MunicipalityID",
        F.col("GiftID").cast("long").alias("GiftID"),
        F.col("DonationAmountYen").cast("long").alias("DonationAmountYen"),
        F.to_timestamp(
            "DonatedAt", "yyyy-MM-dd'T'HH:mm:ss'Z'"
        ).alias("DonatedAt"),
        "PaymentMethod",
    ),
}

for table_name, df in stages.items():
    assert_schema(df, STAGING_SCHEMA_CONTRACT[table_name], table_name)
    assert_no_nulls(df, df.columns, table_name)

assert_unique(stages["stg_prefectures"], ["PrefectureID"], "prefecture PK")
assert_unique(stages["stg_municipalities"], ["MunicipalityID"], "municipality PK")
assert_unique(stages["stg_donors"], ["DonorID"], "donor PK")
assert_unique(stages["stg_categories"], ["CategoryID"], "category PK")
assert_unique(stages["stg_gifts"], ["GiftID"], "gift PK")
assert_unique(stages["stg_businesses"], ["BusinessID"], "business PK")
assert_unique(
    stages["stg_business_gifts"],
    ["BusinessID", "GiftID"],
    "business-gift composite PK",
)
assert_unique(stages["stg_donation_orders"], ["DonationID"], "donation PK")

assert_fk(
    stages["stg_municipalities"], ["PrefectureID"],
    stages["stg_prefectures"], ["PrefectureID"], "municipality prefecture FK",
)
assert_fk(
    stages["stg_donors"], ["PrefectureID"],
    stages["stg_prefectures"], ["PrefectureID"], "donor prefecture FK",
)
assert_fk(
    stages["stg_gifts"], ["CategoryID"],
    stages["stg_categories"], ["CategoryID"], "gift category FK",
)
assert_fk(
    stages["stg_gifts"], ["MunicipalityID"],
    stages["stg_municipalities"], ["MunicipalityID"], "gift municipality FK",
)
assert_fk(
    stages["stg_businesses"], ["PrefectureID"],
    stages["stg_prefectures"], ["PrefectureID"], "business prefecture FK",
)
assert_fk(
    stages["stg_business_gifts"], ["BusinessID"],
    stages["stg_businesses"], ["BusinessID"], "business-gift business FK",
)
assert_fk(
    stages["stg_business_gifts"], ["GiftID"],
    stages["stg_gifts"], ["GiftID"], "business-gift gift FK",
)
assert_fk(
    stages["stg_donation_orders"], ["DonorID"],
    stages["stg_donors"], ["DonorID"], "donation donor FK",
)
assert_fk(
    stages["stg_donation_orders"], ["MunicipalityID"],
    stages["stg_municipalities"], ["MunicipalityID"], "donation municipality FK",
)
assert_fk(
    stages["stg_donation_orders"], ["GiftID"],
    stages["stg_gifts"], ["GiftID"], "donation gift FK",
)

municipality_name_mismatch = (
    stages["stg_municipalities"].alias("m")
    .join(
        stages["stg_prefectures"].alias("p"),
        F.col("m.PrefectureID") == F.col("p.PrefectureID"),
    )
    .where(F.col("m.PrefectureName") != F.col("p.PrefectureName"))
    .limit(1)
    .count()
)
donor_name_mismatch = (
    stages["stg_donors"].alias("d")
    .join(
        stages["stg_prefectures"].alias("p"),
        F.col("d.PrefectureID") == F.col("p.PrefectureID"),
    )
    .where(
        (F.col("d.PrefectureName") != F.col("p.PrefectureName"))
        | (F.col("d.PrefectureNameEn") != F.col("p.PrefectureNameEn"))
    )
    .limit(1)
    .count()
)
business_name_mismatch = (
    stages["stg_businesses"].alias("b")
    .join(
        stages["stg_prefectures"].alias("p"),
        F.col("b.PrefectureID") == F.col("p.PrefectureID"),
    )
    .where(F.col("b.PrefectureName") != F.col("p.PrefectureName"))
    .limit(1)
    .count()
)
donation_gift_mismatch = (
    stages["stg_donation_orders"].alias("d")
    .join(
        stages["stg_gifts"].alias("g"),
        F.col("d.GiftID") == F.col("g.GiftID"),
    )
    .where(F.col("d.MunicipalityID") != F.col("g.MunicipalityID"))
    .limit(1)
    .count()
)
require(municipality_name_mismatch == 0, "Municipality prefecture name mismatch.")
require(donor_name_mismatch == 0, "Donor prefecture label mismatch.")
require(business_name_mismatch == 0, "Business prefecture name mismatch.")
require(donation_gift_mismatch == 0, "Donation municipality differs from gift municipality.")



## Ontology-ready derivations

The eleven `ot_*` frames implement the audited ten-entity, 72-static-property
contract plus relationship key columns and the supplier-gift mapping table.
English labels/search terms and all stable IDs are deterministic.



In [ ]:
prefectures = stages["stg_prefectures"]
municipalities = stages["stg_municipalities"]
donors = stages["stg_donors"]
categories = stages["stg_categories"]
gifts = stages["stg_gifts"]
businesses = stages["stg_businesses"]
business_gifts = stages["stg_business_gifts"]
donations = stages["stg_donation_orders"]

donation_enriched = (
    donations.alias("d")
    .join(
        donors.select("DonorID", "PrefectureID").alias("dn"),
        F.col("d.DonorID") == F.col("dn.DonorID"),
    )
    .join(
        municipalities.select("MunicipalityID", "PrefectureID").alias("m"),
        F.col("d.MunicipalityID") == F.col("m.MunicipalityID"),
    )
    .join(
        gifts.select("GiftID", "CategoryID").alias("g"),
        F.col("d.GiftID") == F.col("g.GiftID"),
    )
    .select(
        F.col("d.DonationID").alias("DonationID"),
        F.col("d.DonorID").alias("DonorID"),
        F.col("d.MunicipalityID").alias("MunicipalityID"),
        F.col("d.GiftID").alias("GiftID"),
        F.col("d.DonationAmountYen").alias("DonationAmountYen"),
        F.col("d.DonatedAt").alias("DonatedAt"),
        F.col("d.PaymentMethod").alias("PaymentMethod"),
        F.col("dn.PrefectureID").alias("ResidencePrefectureID"),
        F.col("m.PrefectureID").alias("RecipientPrefectureID"),
        F.col("g.CategoryID").alias("CategoryID"),
    )
)

received_pref = donation_enriched.groupBy("RecipientPrefectureID").agg(
    F.count(F.lit(1)).cast("long").alias("PrefReceivedStaticCount"),
    F.sum("DonationAmountYen").cast("long").alias("PrefReceivedTotalYen"),
)
resident_pref = donation_enriched.groupBy("ResidencePrefectureID").agg(
    F.count(F.lit(1)).cast("long").alias("PrefResidentStaticCount"),
    F.sum("DonationAmountYen").cast("long").alias("PrefResidentTotalYen"),
)
ot_prefecture = (
    prefectures.select(
        F.col("PrefectureID").alias("PrefectureId"),
        "PrefectureName",
        "PrefectureNameEn",
    )
    .join(
        received_pref,
        F.col("PrefectureId") == F.col("RecipientPrefectureID"),
        "left",
    )
    .join(
        resident_pref,
        F.col("PrefectureId") == F.col("ResidencePrefectureID"),
        "left",
    )
    .fillna(
        0,
        [
            "PrefReceivedStaticCount", "PrefReceivedTotalYen",
            "PrefResidentStaticCount", "PrefResidentTotalYen",
        ],
    )
    .withColumn(
        "PrefReceivedAmountRank",
        F.row_number().over(
            Window.orderBy(F.desc("PrefReceivedTotalYen"), F.asc("PrefectureId"))
        ).cast("long"),
    )
    .withColumn(
        "PrefResidentAmountRank",
        F.row_number().over(
            Window.orderBy(F.desc("PrefResidentTotalYen"), F.asc("PrefectureId"))
        ).cast("long"),
    )
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_prefecture"]])
)

municipality_agg = (
    donation_enriched.groupBy("MunicipalityID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("MunicipalityStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("MunicipalityStaticTotalYen"),
    )
    .withColumnRenamed("MunicipalityID", "__MunicipalityJoinId")
)
ot_municipality = (
    municipalities.select(
        F.col("MunicipalityID").alias("MunicipalityId"),
        "MunicipalityName",
        F.concat_ws(" / ", "MunicipalityName", "PrefectureName").alias(
            "MunicipalityDisplayName"
        ),
        F.col("PrefectureID").alias("PrefectureId"),
    )
    .join(
        municipality_agg,
        F.col("MunicipalityId") == F.col("__MunicipalityJoinId"),
        "left",
    )
    .fillna(0, ["MunicipalityStaticCount", "MunicipalityStaticTotalYen"])
    .withColumn(
        "MunicipalityAmountRank",
        F.row_number().over(
            Window.orderBy(
                F.desc("MunicipalityStaticTotalYen"), F.asc("MunicipalityId")
            )
        ).cast("long"),
    )
    .withColumn(
        "MunicipalityPrefAmountRank",
        F.row_number().over(
            Window.partitionBy("PrefectureId").orderBy(
                F.desc("MunicipalityStaticTotalYen"), F.asc("MunicipalityId")
            )
        ).cast("long"),
    )
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_municipality"]])
)

donor_agg = (
    donation_enriched.groupBy("DonorID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("DonorStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("DonorStaticTotalYen"),
        F.max("DonationAmountYen").cast("long").alias("DonorStaticMaxYen"),
    )
    .withColumnRenamed("DonorID", "__DonorJoinId")
)
ot_donor = (
    donors.select(
        F.col("DonorID").alias("DonorId"),
        "DonorName",
        F.concat(
            "DonorName", F.lit(" / Donor "), F.col("DonorID").cast("string")
        ).alias("DonorDisplayName"),
        F.col("Age").alias("DonorAge"),
        F.col("Occupation").alias("DonorOccupation"),
        F.col("OccupationEn").alias("DonorOccupationEn"),
        F.col("PrefectureID").alias("PrefectureId"),
    )
    .join(donor_agg, F.col("DonorId") == F.col("__DonorJoinId"), "left")
    .fillna(
        0,
        ["DonorStaticCount", "DonorStaticTotalYen", "DonorStaticMaxYen"],
    )
    .withColumn(
        "DonorOverallAmountRank",
        F.row_number().over(
            Window.orderBy(F.desc("DonorStaticTotalYen"), F.asc("DonorId"))
        ).cast("long"),
    )
    .withColumn(
        "DonorResidenceAmountRank",
        F.row_number().over(
            Window.partitionBy("PrefectureId").orderBy(
                F.desc("DonorStaticTotalYen"), F.asc("DonorId")
            )
        ).cast("long"),
    )
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_donor"]])
)

category_agg = (
    donation_enriched.groupBy("CategoryID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("CategoryStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("CategoryStaticTotalYen"),
    )
    .withColumnRenamed("CategoryID", "__CategoryJoinId")
)
category_labels = categories.select(
    F.col("CategoryID").alias("CategoryId"),
    "CategoryName",
    F.col("CategoryNameEn"),
)
ot_gift_category = (
    category_labels.join(
        category_agg, F.col("CategoryId") == F.col("__CategoryJoinId"), "left"
    )
    .fillna(0, ["CategoryStaticCount", "CategoryStaticTotalYen"])
    .withColumn(
        "CategorySearchTerms",
        F.concat_ws(
            " | ", "CategoryName", "CategoryNameEn", F.lit("gift category")
        ),
    )
    .withColumn(
        "CategoryAmountRank",
        F.row_number().over(
            Window.orderBy(F.desc("CategoryStaticTotalYen"), F.asc("CategoryId"))
        ).cast("long"),
    )
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_gift_category"]])
)

gift_agg = (
    donation_enriched.groupBy("GiftID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("GiftStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("GiftStaticTotalYen"),
    )
    .withColumnRenamed("GiftID", "__GiftJoinId")
)
ot_gift = (
    gifts.alias("g")
    .join(
        category_labels.alias("c"),
        F.col("g.CategoryID") == F.col("c.CategoryId"),
    )
    .select(
        F.col("g.GiftID").alias("GiftId"),
        F.col("g.GiftName").alias("GiftName"),
        F.concat(
            F.col("g.GiftName"),
            F.lit(" / Gift "),
            F.col("g.GiftID").cast("string"),
        ).alias("GiftDisplayName"),
        F.concat_ws(
            " | ",
            F.col("g.GiftName"),
            F.col("g.GiftNameEn"),
            F.col("c.CategoryName"),
            F.col("c.CategoryNameEn"),
        ).alias("GiftSearchTerms"),
        F.col("g.Notes").alias("GiftNotes"),
        F.col("g.CategoryID").alias("CategoryId"),
        F.col("g.MunicipalityID").alias("MunicipalityId"),
    )
    .join(gift_agg, F.col("GiftId") == F.col("__GiftJoinId"), "left")
    .fillna(0, ["GiftStaticCount", "GiftStaticTotalYen"])
    .withColumn(
        "GiftAmountRank",
        F.row_number().over(
            Window.orderBy(F.desc("GiftStaticTotalYen"), F.asc("GiftId"))
        ).cast("long"),
    )
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_gift"]])
)

supplier_agg = business_gifts.groupBy("BusinessID").agg(
    F.count(F.lit(1)).cast("long").alias("SupplierProvidedGiftCount")
)
ot_supplier = (
    businesses.select(
        F.col("BusinessID").alias("SupplierId"),
        F.col("BusinessName").alias("SupplierName"),
        F.concat(
            "BusinessName",
            F.lit(" / Supplier "),
            F.col("BusinessID").cast("string"),
        ).alias("SupplierDisplayName"),
        F.col("BusinessType").alias("SupplierType"),
        F.col("BusinessTypeEn").alias("SupplierTypeEn"),
        F.col("PrefectureID").alias("PrefectureId"),
    )
    .join(supplier_agg, F.col("SupplierId") == F.col("BusinessID"), "left")
    .fillna(0, ["SupplierProvidedGiftCount"])
    .select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_supplier"]])
)

ot_supplier_gift = business_gifts.select(
    F.col("BusinessID").alias("SupplierId"),
    F.col("GiftID").alias("GiftId"),
)

payment_method_en = literal_map(
    {
        "クレジットカード": "Credit card",
        "銀行振込": "Bank transfer",
        "コンビニ決済": "Convenience store payment",
        "電子決済": "Electronic payment",
    }
)
donation_jst = F.from_utc_timestamp(F.col("DonatedAt"), "Asia/Tokyo")
ot_donation = donations.select(
    F.col("DonationID").alias("DonationId"),
    F.concat(
        F.lit("Donation "), F.col("DonationID").cast("string")
    ).alias("DonationDisplayName"),
    "DonationAmountYen",
    F.col("DonatedAt").alias("DonatedAtUtc"),
    F.concat(F.date_format(donation_jst, "yyyy-MM-dd HH:mm:ss"), F.lit(" JST")).alias(
        "DonatedAtJstText"
    ),
    F.date_format(donation_jst, "yyyy-MM-dd").alias("DonationDateJst"),
    F.date_format(donation_jst, "yyyy-MM").alias("DonationYearMonthJst"),
    F.concat(
        F.year(donation_jst).cast("string"),
        F.lit("年"),
        F.month(donation_jst).cast("string"),
        F.lit("月"),
    ).alias("DonationYearMonthJaShort"),
    F.col("PaymentMethod").alias("DonationPaymentMethod"),
    F.element_at(payment_method_en, F.col("PaymentMethod")).alias(
        "DonationPaymentMethodEn"
    ),
    F.lit(DONATION_DATA_LAYER).alias("DonationDataLayer"),
    F.col("DonorID").alias("DonorId"),
    F.col("MunicipalityID").alias("MunicipalityId"),
    F.col("GiftID").alias("GiftId"),
).select(*[name for name, _ in OUTPUT_SCHEMA_CONTRACT["ot_donation"]])

ot_mun_category_metric = (
    donation_enriched.groupBy("MunicipalityID", "CategoryID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("MunCategoryStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("MunCategoryTotalYen"),
    )
    .withColumn(
        "MunCategoryMetricId",
        F.concat(
            F.col("MunicipalityID"),
            F.lit("-"),
            F.lpad(F.col("CategoryID").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "MunCategoryAmountRank",
        F.row_number().over(
            Window.partitionBy("MunicipalityID").orderBy(
                F.desc("MunCategoryTotalYen"), F.asc("CategoryID")
            )
        ).cast("long"),
    )
    .select(
        "MunCategoryMetricId",
        "MunCategoryStaticCount",
        "MunCategoryTotalYen",
        "MunCategoryAmountRank",
        F.col("MunicipalityID").alias("MunicipalityId"),
        F.col("CategoryID").alias("CategoryId"),
    )
)

ot_pref_category_metric = (
    donation_enriched.groupBy("RecipientPrefectureID", "CategoryID")
    .agg(
        F.count(F.lit(1)).cast("long").alias("PrefCategoryStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("PrefCategoryTotalYen"),
    )
    .withColumn(
        "PrefCategoryMetricId",
        F.concat(
            F.lpad(F.col("RecipientPrefectureID").cast("string"), 2, "0"),
            F.lit("-"),
            F.lpad(F.col("CategoryID").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "PrefCategoryAmountRank",
        F.row_number().over(
            Window.partitionBy("RecipientPrefectureID").orderBy(
                F.desc("PrefCategoryTotalYen"), F.asc("CategoryID")
            )
        ).cast("long"),
    )
    .select(
        "PrefCategoryMetricId",
        "PrefCategoryStaticCount",
        "PrefCategoryTotalYen",
        "PrefCategoryAmountRank",
        F.col("RecipientPrefectureID").alias("PrefectureId"),
        F.col("CategoryID").alias("CategoryId"),
    )
)

ot_pref_donation_flow = (
    donation_enriched.groupBy(
        "ResidencePrefectureID", "RecipientPrefectureID"
    )
    .agg(
        F.count(F.lit(1)).cast("long").alias("PrefFlowStaticCount"),
        F.sum("DonationAmountYen").cast("long").alias("PrefFlowTotalYen"),
    )
    .withColumn(
        "PrefDonationFlowId",
        F.concat(
            F.lpad(F.col("ResidencePrefectureID").cast("string"), 2, "0"),
            F.lit("-"),
            F.lpad(F.col("RecipientPrefectureID").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "PrefFlowOriginRank",
        F.row_number().over(
            Window.partitionBy("ResidencePrefectureID").orderBy(
                F.desc("PrefFlowTotalYen"), F.asc("RecipientPrefectureID")
            )
        ).cast("long"),
    )
    .withColumn(
        "PrefFlowDestinationRank",
        F.row_number().over(
            Window.partitionBy("RecipientPrefectureID").orderBy(
                F.desc("PrefFlowTotalYen"), F.asc("ResidencePrefectureID")
            )
        ).cast("long"),
    )
    .select(
        "PrefDonationFlowId",
        "PrefFlowStaticCount",
        "PrefFlowTotalYen",
        "PrefFlowOriginRank",
        "PrefFlowDestinationRank",
        F.col("ResidencePrefectureID").alias("ResidencePrefectureId"),
        F.col("RecipientPrefectureID").alias("RecipientPrefectureId"),
    )
)

outputs = {
    "ot_prefecture": ot_prefecture,
    "ot_municipality": ot_municipality,
    "ot_donor": ot_donor,
    "ot_gift_category": ot_gift_category,
    "ot_gift": ot_gift,
    "ot_supplier": ot_supplier,
    "ot_supplier_gift": ot_supplier_gift,
    "ot_donation": ot_donation,
    "ot_mun_category_metric": ot_mun_category_metric,
    "ot_pref_category_metric": ot_pref_category_metric,
    "ot_pref_donation_flow": ot_pref_donation_flow,
}



## In-memory contract and accuracy gates

These gates run before any table write and are reused for temporary and final Delta
tables. Totals reconcile at Donation, Gift, Municipality, Prefecture, Category,
municipality-category, prefecture-category, and prefecture-flow grains.



In [ ]:
STAGE_PRIMARY_KEYS = {
    "stg_prefectures": ["PrefectureID"],
    "stg_municipalities": ["MunicipalityID"],
    "stg_donors": ["DonorID"],
    "stg_categories": ["CategoryID"],
    "stg_gifts": ["GiftID"],
    "stg_businesses": ["BusinessID"],
    "stg_business_gifts": ["BusinessID", "GiftID"],
    "stg_donation_orders": ["DonationID"],
}
OUTPUT_PRIMARY_KEYS = {
    "ot_prefecture": ["PrefectureId"],
    "ot_municipality": ["MunicipalityId"],
    "ot_donor": ["DonorId"],
    "ot_gift_category": ["CategoryId"],
    "ot_gift": ["GiftId"],
    "ot_supplier": ["SupplierId"],
    "ot_supplier_gift": ["SupplierId", "GiftId"],
    "ot_donation": ["DonationId"],
    "ot_mun_category_metric": ["MunCategoryMetricId"],
    "ot_pref_category_metric": ["PrefCategoryMetricId"],
    "ot_pref_donation_flow": ["PrefDonationFlowId"],
}
AUDIT_SCHEMA_CONTRACT = [
    ("RunId", "string"),
    ("ParticipantId", "string"),
    ("NotebookVersion", "string"),
    ("DatasetVersion", "string"),
    ("ManifestVersion", "bigint"),
    ("ChecksumContract", "string"),
    ("InputDirectory", "string"),
    ("SourceFile", "string"),
    ("ExpectedSha256", "string"),
    ("ActualSha256", "string"),
    ("ExpectedRows", "bigint"),
    ("ActualRows", "bigint"),
    ("SourceBytes", "bigint"),
    ("ValidationStatus", "string"),
    ("ValidatedAtUtc", "timestamp"),
]


def scalar_sum(df: DataFrame, column: str) -> int:
    value = df.agg(F.sum(column).alias("value")).first()["value"]
    return int(value or 0)


def validate_stage_frames(stage_frames: dict[str, DataFrame]) -> None:
    require(list(stage_frames) == STAGE_TABLE_ORDER, "Unexpected staging table order.")
    for file_name, contract in INPUT_CONTRACT.items():
        table_name = contract["stage"]
        df = stage_frames[table_name]
        assert_schema(df, STAGING_SCHEMA_CONTRACT[table_name], table_name)
        assert_row_count(df, contract["rows"], table_name)
        assert_no_nulls(df, df.columns, table_name)
        assert_unique(df, STAGE_PRIMARY_KEYS[table_name], f"{table_name} PK")


def validate_output_frames(output_frames: dict[str, DataFrame]) -> None:
    require(list(output_frames) == OUTPUT_TABLE_ORDER, "Unexpected output table order.")
    for table_name, expected_rows in OUTPUT_ROW_COUNTS.items():
        df = output_frames[table_name]
        assert_schema(df, OUTPUT_SCHEMA_CONTRACT[table_name], table_name)
        assert_row_count(df, expected_rows, table_name)
        assert_no_nulls(df, df.columns, table_name)
        assert_unique(df, OUTPUT_PRIMARY_KEYS[table_name], f"{table_name} PK")

    assert_fk(
        output_frames["ot_municipality"], ["PrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "ontology municipality prefecture FK",
    )
    assert_fk(
        output_frames["ot_donor"], ["PrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "ontology donor prefecture FK",
    )
    assert_fk(
        output_frames["ot_gift"], ["CategoryId"],
        output_frames["ot_gift_category"], ["CategoryId"], "ontology gift category FK",
    )
    assert_fk(
        output_frames["ot_gift"], ["MunicipalityId"],
        output_frames["ot_municipality"], ["MunicipalityId"], "ontology gift municipality FK",
    )
    assert_fk(
        output_frames["ot_supplier"], ["PrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "ontology supplier prefecture FK",
    )
    assert_fk(
        output_frames["ot_supplier_gift"], ["SupplierId"],
        output_frames["ot_supplier"], ["SupplierId"], "ontology supplier-gift supplier FK",
    )
    assert_fk(
        output_frames["ot_supplier_gift"], ["GiftId"],
        output_frames["ot_gift"], ["GiftId"], "ontology supplier-gift gift FK",
    )
    assert_fk(
        output_frames["ot_donation"], ["DonorId"],
        output_frames["ot_donor"], ["DonorId"], "ontology donation donor FK",
    )
    assert_fk(
        output_frames["ot_donation"], ["MunicipalityId"],
        output_frames["ot_municipality"], ["MunicipalityId"], "ontology donation municipality FK",
    )
    assert_fk(
        output_frames["ot_donation"], ["GiftId"],
        output_frames["ot_gift"], ["GiftId"], "ontology donation gift FK",
    )
    assert_fk(
        output_frames["ot_mun_category_metric"], ["MunicipalityId"],
        output_frames["ot_municipality"], ["MunicipalityId"], "municipality metric FK",
    )
    assert_fk(
        output_frames["ot_mun_category_metric"], ["CategoryId"],
        output_frames["ot_gift_category"], ["CategoryId"], "municipality metric category FK",
    )
    assert_fk(
        output_frames["ot_pref_category_metric"], ["PrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "prefecture metric FK",
    )
    assert_fk(
        output_frames["ot_pref_category_metric"], ["CategoryId"],
        output_frames["ot_gift_category"], ["CategoryId"], "prefecture metric category FK",
    )
    assert_fk(
        output_frames["ot_pref_donation_flow"], ["ResidencePrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "flow origin FK",
    )
    assert_fk(
        output_frames["ot_pref_donation_flow"], ["RecipientPrefectureId"],
        output_frames["ot_prefecture"], ["PrefectureId"], "flow destination FK",
    )

    donation = output_frames["ot_donation"]
    require(
        donation.select("DonationId").distinct().count() == 80000,
        "Donation distinct ID gate failed.",
    )
    require(
        scalar_sum(donation, "DonationAmountYen") == 1344099000,
        "Donation total gate failed.",
    )
    require(
        donation.select("DonationDataLayer").distinct().collect()[0][0]
        == DONATION_DATA_LAYER
        and donation.select("DonationDataLayer").distinct().count() == 1,
        "DonationDataLayer gate failed.",
    )

    reconciliation_columns = {
        "ot_donor": ("DonorStaticCount", "DonorStaticTotalYen"),
        "ot_gift": ("GiftStaticCount", "GiftStaticTotalYen"),
        "ot_municipality": ("MunicipalityStaticCount", "MunicipalityStaticTotalYen"),
        "ot_prefecture": ("PrefReceivedStaticCount", "PrefReceivedTotalYen"),
        "ot_gift_category": ("CategoryStaticCount", "CategoryStaticTotalYen"),
        "ot_mun_category_metric": ("MunCategoryStaticCount", "MunCategoryTotalYen"),
        "ot_pref_category_metric": ("PrefCategoryStaticCount", "PrefCategoryTotalYen"),
        "ot_pref_donation_flow": ("PrefFlowStaticCount", "PrefFlowTotalYen"),
    }
    for table_name, (count_column, amount_column) in reconciliation_columns.items():
        require(
            scalar_sum(output_frames[table_name], count_column) == 80000,
            f"{table_name}: count reconciliation failed.",
        )
        require(
            scalar_sum(output_frames[table_name], amount_column) == 1344099000,
            f"{table_name}: amount reconciliation failed.",
        )
    require(
        scalar_sum(output_frames["ot_prefecture"], "PrefResidentStaticCount") == 80000,
        "Resident prefecture count reconciliation failed.",
    )
    require(
        scalar_sum(output_frames["ot_prefecture"], "PrefResidentTotalYen")
        == 1344099000,
        "Resident prefecture amount reconciliation failed.",
    )

    tokyo_rows = (
        output_frames["ot_donor"]
        .where(F.col("DonorId") == 2005075)
        .select(
            "DonorName", "PrefectureId", "DonorStaticCount",
            "DonorStaticTotalYen", "DonorStaticMaxYen",
            "DonorResidenceAmountRank",
        )
        .collect()
    )
    require(len(tokyo_rows) == 1, "Tokyo donor gate row is missing.")
    tokyo = tokyo_rows[0]
    require(tokyo["DonorName"] == "野口啓介", "Tokyo donor name gate failed.")
    require(int(tokyo["PrefectureId"]) == 13, "Tokyo donor residence gate failed.")
    require(int(tokyo["DonorStaticCount"]) == 8, "Tokyo donor count gate failed.")
    require(int(tokyo["DonorStaticTotalYen"]) == 383000, "Tokyo donor total gate failed.")
    require(int(tokyo["DonorStaticMaxYen"]) == 300000, "Tokyo donor max gate failed.")
    require(int(tokyo["DonorResidenceAmountRank"]) == 1, "Tokyo donor rank gate failed.")

    require(
        output_frames["ot_donor"].where(F.col("DonorStaticCount") == 0).count() == 18,
        "Donors-without-donations gate failed.",
    )
    require(
        output_frames["ot_municipality"]
        .where(F.col("MunicipalityStaticCount") == 0)
        .count()
        == 10,
        "Municipalities-without-donations gate failed.",
    )
    require(
        output_frames["ot_supplier"]
        .where(F.col("SupplierProvidedGiftCount") == 0)
        .count()
        == 0,
        "Suppliers-without-gifts gate failed.",
    )

    require(
        not has_rows(
            output_frames["ot_municipality"],
            ~F.col("MunicipalityId").rlike(r"^[0-9]{6}$"),
        ),
        "Ontology MunicipalityId lost the six-character contract.",
    )
    require(
        not has_rows(
            output_frames["ot_mun_category_metric"],
            ~F.col("MunCategoryMetricId").rlike(r"^[0-9]{6}-[0-9]{2}$"),
        ),
        "Municipality-category stable ID gate failed.",
    )
    require(
        not has_rows(
            output_frames["ot_pref_category_metric"],
            ~F.col("PrefCategoryMetricId").rlike(r"^[0-9]{2}-[0-9]{2}$"),
        ),
        "Prefecture-category stable ID gate failed.",
    )
    require(
        not has_rows(
            output_frames["ot_pref_donation_flow"],
            ~F.col("PrefDonationFlowId").rlike(r"^[0-9]{2}-[0-9]{2}$"),
        ),
        "Prefecture-flow stable ID gate failed.",
    )

    assert_rank_matches_order(
        output_frames["ot_prefecture"],
        "PrefReceivedAmountRank", "PrefReceivedTotalYen", ["PrefectureId"], [],
        "prefecture received rank",
    )
    assert_rank_matches_order(
        output_frames["ot_prefecture"],
        "PrefResidentAmountRank", "PrefResidentTotalYen", ["PrefectureId"], [],
        "prefecture resident rank",
    )
    assert_rank_matches_order(
        output_frames["ot_municipality"],
        "MunicipalityAmountRank", "MunicipalityStaticTotalYen", ["MunicipalityId"], [],
        "municipality overall rank",
    )
    assert_rank_matches_order(
        output_frames["ot_municipality"],
        "MunicipalityPrefAmountRank", "MunicipalityStaticTotalYen", ["MunicipalityId"],
        ["PrefectureId"], "municipality prefecture rank",
    )
    assert_rank_matches_order(
        output_frames["ot_donor"],
        "DonorOverallAmountRank", "DonorStaticTotalYen", ["DonorId"], [],
        "donor overall rank",
    )
    assert_rank_matches_order(
        output_frames["ot_donor"],
        "DonorResidenceAmountRank", "DonorStaticTotalYen", ["DonorId"],
        ["PrefectureId"], "donor residence rank",
    )
    assert_rank_matches_order(
        output_frames["ot_gift_category"],
        "CategoryAmountRank", "CategoryStaticTotalYen", ["CategoryId"], [],
        "category rank",
    )
    assert_rank_matches_order(
        output_frames["ot_gift"],
        "GiftAmountRank", "GiftStaticTotalYen", ["GiftId"], [], "gift rank",
    )
    assert_rank_matches_order(
        output_frames["ot_mun_category_metric"],
        "MunCategoryAmountRank", "MunCategoryTotalYen", ["CategoryId"],
        ["MunicipalityId"], "municipality-category rank",
    )
    assert_rank_matches_order(
        output_frames["ot_pref_category_metric"],
        "PrefCategoryAmountRank", "PrefCategoryTotalYen", ["CategoryId"],
        ["PrefectureId"], "prefecture-category rank",
    )
    assert_rank_matches_order(
        output_frames["ot_pref_donation_flow"],
        "PrefFlowOriginRank", "PrefFlowTotalYen", ["RecipientPrefectureId"],
        ["ResidencePrefectureId"], "flow origin rank",
    )
    assert_rank_matches_order(
        output_frames["ot_pref_donation_flow"],
        "PrefFlowDestinationRank", "PrefFlowTotalYen", ["ResidencePrefectureId"],
        ["RecipientPrefectureId"], "flow destination rank",
    )


validate_stage_frames(stages)
validate_output_frames(outputs)

validated_at_utc = datetime.now(timezone.utc).replace(tzinfo=None)
audit_schema = T.StructType(
    [
        T.StructField("RunId", T.StringType(), False),
        T.StructField("ParticipantId", T.StringType(), False),
        T.StructField("NotebookVersion", T.StringType(), False),
        T.StructField("DatasetVersion", T.StringType(), False),
        T.StructField("ManifestVersion", T.LongType(), False),
        T.StructField("ChecksumContract", T.StringType(), False),
        T.StructField("InputDirectory", T.StringType(), False),
        T.StructField("SourceFile", T.StringType(), False),
        T.StructField("ExpectedSha256", T.StringType(), False),
        T.StructField("ActualSha256", T.StringType(), False),
        T.StructField("ExpectedRows", T.LongType(), False),
        T.StructField("ActualRows", T.LongType(), False),
        T.StructField("SourceBytes", T.LongType(), False),
        T.StructField("ValidationStatus", T.StringType(), False),
        T.StructField("ValidatedAtUtc", T.TimestampType(), False),
    ]
)
audit_rows = []
for file_name, contract in INPUT_CONTRACT.items():
    audit_rows.append(
        (
            RUN_ID,
            PARTICIPANT_ID,
            NOTEBOOK_VERSION,
            DATASET_VERSION,
            MANIFEST_VERSION,
            CHECKSUM_CONTRACT,
            INPUT_DIR,
            file_name,
            contract["sha256"],
            actual_hashes[file_name],
            int(contract["rows"]),
            int(raw[file_name].count()),
            int(byte_sizes[file_name]),
            "VALIDATED_TEMP",
            validated_at_utc,
        )
    )
audit_frame = spark.createDataFrame(audit_rows, schema=audit_schema)
assert_schema(audit_frame, AUDIT_SCHEMA_CONTRACT, AUDIT_TABLE)
assert_row_count(audit_frame, 8, AUDIT_TABLE)
assert_unique(audit_frame, ["RunId", "SourceFile"], "audit manifest PK")



## Run-scoped Delta materialization

All 20 candidate tables (8 staging, 11 ontology-ready, and 1 audit manifest) are
first written under names unique to this run. The notebook rereads and validates
those Delta tables before any final table is changed, and the publication path then
reads only those validated Delta tables, so the bytes that are published are exactly
the bytes that passed validation.



In [ ]:
temp_prefix = f"tmp_furusato_v270_p{PARTICIPANT_ID}_{uuid.uuid4().hex[:12]}"
TEMP_TABLE_MAP = {
    final_name: f"{temp_prefix}_{final_name}" for final_name in FINAL_TABLE_ORDER
}
candidate_frames = {
    **stages,
    **outputs,
    AUDIT_TABLE: audit_frame,
}
written_temp_tables = []

try:
    for final_name in FINAL_TABLE_ORDER:
        temp_name = TEMP_TABLE_MAP[final_name]
        (
            candidate_frames[final_name]
            .write.format("delta")
            .mode("errorifexists")
            .saveAsTable(temp_name)
        )
        written_temp_tables.append(temp_name)

    temp_stages = {
        name: spark.table(TEMP_TABLE_MAP[name]) for name in STAGE_TABLE_ORDER
    }
    temp_outputs = {
        name: spark.table(TEMP_TABLE_MAP[name]) for name in OUTPUT_TABLE_ORDER
    }
    temp_audit = spark.table(TEMP_TABLE_MAP[AUDIT_TABLE])

    validate_stage_frames(temp_stages)
    validate_output_frames(temp_outputs)
    assert_schema(temp_audit, AUDIT_SCHEMA_CONTRACT, AUDIT_TABLE)
    assert_row_count(temp_audit, 8, AUDIT_TABLE)
    require(
        temp_audit.where(F.col("ValidationStatus") != "VALIDATED_TEMP").count() == 0,
        "Temporary audit manifest status gate failed.",
    )
    candidate_frames = {
        name: spark.table(TEMP_TABLE_MAP[name]) for name in FINAL_TABLE_ORDER
    }
except Exception:
    cleanup_errors = drop_tables_best_effort(written_temp_tables)
    if cleanup_errors:
        print("Temporary cleanup warnings:", cleanup_errors)
    raise



## Atomic per-table generation CAS and global readiness

Every final managed table contains the unbound technical column `_WorkshopGenerationId`.
The 72 Ontology properties and participant-facing validation projections exclude this column.
Each complete candidate table is applied with one Delta `MERGE` transaction keyed by its
documented PK. Matched rows are updated, new keys inserted, and removed prior-generation keys
deleted. The MERGE source carries both the candidate rows and the exact `VERSION AS OF` pre-publish
snapshot. Every matched, missing, inserted, and removed key is compared inside the same Delta
transaction; generation-preserving edits and snapshot drift evaluate `raise_error` before mutation.

The merge includes matched and not-matched-by-source paths for the complete table. It therefore
reads/touches the prior target set; under Fabric Delta Serializable optimistic concurrency, a
commit after the transaction snapshot causes the merge to fail rather than silently serialize
over it. Rollback uses the identical snapshot-content CAS with previous rows read from the recorded
`VERSION AS OF` snapshot and requires the failed generation plus exact candidate content. `RESTORE` and final
`saveAsTable` overwrite paths are not used. Empty new tables are created under the exclusive
lease with `errorifexists`, then populated by the same guarded merge.

The persistent publish-control lease gates `Preparing`, `Publishing`, `Recovering`, `Ready`,
`Failed`, and `Stale`. Durable per-table intent is recorded before every MERGE. Exceptions and
stale recovery reconcile retained candidate snapshots, Delta history, and generation values before
rolling back every committed target; unexplained drift retains recovery ownership. Consumers must
reject every non-Ready state. Immutable Ready evidence also proves historical success when the next
generation has already acquired the lease. A hard stop remains
non-Ready. Cross-table multi-ACID atomicity is not claimed; the per-table CAS prevents destructive
race overwrites. Optimization is deferred and never runs in the fenced publication path.


In [ ]:
PUBLISH_CONTROL_SCHEMA_CONTRACT = [
    ("ControlKey", "string"),
    ("RecordType", "string"),
    ("TableName", "string"),
    ("Generation", "bigint"),
    ("PublicationRunId", "string"),
    ("OwnerRunId", "string"),
    ("PublishState", "string"),
    ("LeaseAcquiredAtUtc", "timestamp"),
    ("LeaseExpiresAtUtc", "timestamp"),
    ("PrePublishVersion", "bigint"),
    ("ExpectedCurrentVersion", "bigint"),
    ("PublishedVersion", "bigint"),
    ("RollbackVersion", "bigint"),
    ("CandidateTableName", "string"),
    ("MutationState", "string"),
    ("UpdatedAtUtc", "timestamp"),
    ("FailureReason", "string"),
]
PUBLISH_CONTROL_SCHEMA = T.StructType(
    [
        T.StructField("ControlKey", T.StringType(), False),
        T.StructField("RecordType", T.StringType(), False),
        T.StructField("TableName", T.StringType(), False),
        T.StructField("Generation", T.LongType(), False),
        T.StructField("PublicationRunId", T.StringType(), False),
        T.StructField("OwnerRunId", T.StringType(), True),
        T.StructField("PublishState", T.StringType(), False),
        T.StructField("LeaseAcquiredAtUtc", T.TimestampType(), True),
        T.StructField("LeaseExpiresAtUtc", T.TimestampType(), True),
        T.StructField("PrePublishVersion", T.LongType(), False),
        T.StructField("ExpectedCurrentVersion", T.LongType(), False),
        T.StructField("PublishedVersion", T.LongType(), False),
        T.StructField("RollbackVersion", T.LongType(), False),
        T.StructField("CandidateTableName", T.StringType(), True),
        T.StructField("MutationState", T.StringType(), True),
        T.StructField("UpdatedAtUtc", T.TimestampType(), False),
        T.StructField("FailureReason", T.StringType(), True),
    ]
)
ACTIVE_PUBLISH_STATES = ("Preparing", "Publishing", "Recovering")
EXPECTED_GENERATION_RECORDS = len(FINAL_TABLE_ORDER) + 1


def utc_now() -> datetime:
    return datetime.now(timezone.utc).replace(tzinfo=None)


def control_delta() -> DeltaTable:
    return DeltaTable.forName(spark, PUBLISH_CONTROL_TABLE)


def control_rows() -> DataFrame:
    return spark.table(PUBLISH_CONTROL_TABLE)


def current_lease_row():
    rows = (
        control_rows()
        .where(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Lease")
            & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
        )
        .collect()
    )
    require(len(rows) == 1, "Publish control must contain exactly one lease row.")
    return rows[0]


def generation_rows(generation: int) -> DataFrame:
    return control_rows().where(
        (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
        & (F.col("Generation") == generation)
    )


def ensure_publish_control_table() -> None:
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {quoted_table(PUBLISH_CONTROL_TABLE)} (
            ControlKey STRING,
            RecordType STRING,
            TableName STRING,
            Generation BIGINT,
            PublicationRunId STRING,
            OwnerRunId STRING,
            PublishState STRING,
            LeaseAcquiredAtUtc TIMESTAMP,
            LeaseExpiresAtUtc TIMESTAMP,
            PrePublishVersion BIGINT,
            ExpectedCurrentVersion BIGINT,
            PublishedVersion BIGINT,
            RollbackVersion BIGINT,
            CandidateTableName STRING,
            MutationState STRING,
            UpdatedAtUtc TIMESTAMP,
            FailureReason STRING
        ) USING DELTA
        """
    )
    existing_columns = set(spark.table(PUBLISH_CONTROL_TABLE).columns)
    for column_name, column_type, after_column in (
        ("CandidateTableName", "STRING", "RollbackVersion"),
        ("MutationState", "STRING", "CandidateTableName"),
    ):
        if column_name not in existing_columns:
            spark.sql(
                f"ALTER TABLE {quoted_table(PUBLISH_CONTROL_TABLE)} ADD COLUMNS "
                f"({quoted_column(column_name)} {column_type} "
                f"AFTER {quoted_column(after_column)})"
            )
            existing_columns.add(column_name)
    assert_schema(
        spark.table(PUBLISH_CONTROL_TABLE),
        PUBLISH_CONTROL_SCHEMA_CONTRACT,
        PUBLISH_CONTROL_TABLE,
    )
    now = utc_now()
    initial = spark.createDataFrame(
        [
            (
                PUBLISH_CONTROL_KEY,
                "Lease",
                PUBLISH_CONTROL_RECORD,
                0,
                "",
                None,
                "Failed",
                None,
                None,
                -1,
                -1,
                -1,
                -1,
                None,
                "NotApplicable",
                now,
                "Uninitialized; first successful publication is required.",
            )
        ],
        schema=PUBLISH_CONTROL_SCHEMA,
    )
    try:
        (
            control_delta()
            .alias("target")
            .merge(
                initial.alias("source"),
                "target.ControlKey = source.ControlKey "
                "AND target.RecordType = 'Lease' "
                "AND target.TableName = source.TableName",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
    except Exception:
        existing = (
            control_rows()
            .where(
                (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
                & (F.col("RecordType") == "Lease")
                & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
            )
            .limit(1)
            .count()
        )
        if existing != 1:
            raise
    current_lease_row()


def mark_expired_lease_stale(lease) -> None:
    now = utc_now()
    generation = int(lease["Generation"])
    owner = lease["OwnerRunId"]
    state = lease["PublishState"]
    expiry = lease["LeaseExpiresAtUtc"]
    require(owner is not None, "Cannot mark an unowned lease stale.")
    require(state in ACTIVE_PUBLISH_STATES, f"Lease state {state} cannot become Stale.")
    require(expiry is not None and expiry <= now, "Lease has not expired.")
    reason = (
        f"Lease for generation {generation} expired while owned by {owner}; "
        "explicit operator recovery is required."
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Lease")
            & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
            & (F.col("Generation") == generation)
            & (F.col("OwnerRunId") == owner)
            & (F.col("PublishState") == state)
            & (F.col("LeaseExpiresAtUtc") <= F.lit(now))
        ),
        set={
            "PublishState": F.lit("Stale"),
            "UpdatedAtUtc": F.lit(now),
            "FailureReason": F.lit(reason),
        },
    )
    stale_lease = current_lease_row()
    require(
        int(stale_lease["Generation"]) == generation
        and stale_lease["OwnerRunId"] == owner
        and stale_lease["PublishState"] == "Stale",
        "Expired lease stale transition lost a compare-before-update race.",
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("Generation") == generation)
            & (F.col("PublicationRunId") == lease["PublicationRunId"])
            & (F.col("OwnerRunId") == owner)
            & F.col("PublishState").isin(*ACTIVE_PUBLISH_STATES)
        ),
        set={
            "PublishState": F.lit("Stale"),
            "UpdatedAtUtc": F.lit(now),
            "FailureReason": F.lit(reason),
        },
    )


def recover_stale_lease_explicitly(lease) -> None:
    require(OPERATOR_RECOVER_STALE_LEASE, "Stale lease recovery was not explicitly enabled.")
    require(
        lease["PublishState"] == "Stale",
        "Explicit stale recovery requires a persisted Stale lease.",
    )
    require(
        lease["OwnerRunId"] == STALE_LEASE_OWNER_RUN_ID,
        "Stale owner acknowledgement does not match the control record.",
    )
    require(
        int(lease["Generation"]) == STALE_LEASE_GENERATION,
        "Stale generation acknowledgement does not match the control record.",
    )
    require(
        lease["LeaseExpiresAtUtc"] is not None
        and lease["LeaseExpiresAtUtc"] <= utc_now(),
        "Operator recovery cannot clear an unexpired lease.",
    )
    now = utc_now()
    expiry = now + timedelta(minutes=PUBLISH_LEASE_MINUTES)
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("Generation") == STALE_LEASE_GENERATION)
            & (F.col("PublicationRunId") == lease["PublicationRunId"])
            & (F.col("OwnerRunId") == STALE_LEASE_OWNER_RUN_ID)
            & F.col("PublishState").isin("Stale", "Preparing", "Publishing")
        ),
        set={
            "OwnerRunId": F.lit(RUN_ID),
            "PublishState": F.lit("Recovering"),
            "LeaseExpiresAtUtc": F.lit(expiry),
            "UpdatedAtUtc": F.lit(now),
            "FailureReason": F.lit(
                f"Recovery ownership taken by {RUN_ID}; stale kernel must remain stopped."
            ),
        },
    )
    recovery_records = generation_rows(STALE_LEASE_GENERATION).collect()
    require(
        len(recovery_records) == EXPECTED_GENERATION_RECORDS
        and all(
            row["PublicationRunId"] == lease["PublicationRunId"]
            and row["OwnerRunId"] == RUN_ID
            and row["PublishState"] == "Recovering"
            for row in recovery_records
        ),
        "Exclusive stale recovery ownership was not acquired for every evidence row.",
    )
    previous_ready_generation = latest_ready_generation(STALE_LEASE_GENERATION)
    previous_generation_id = (
        workshop_generation_id(previous_ready_generation)
        if previous_ready_generation is not None
        else NO_READY_GENERATION_ID
    )
    reconcile_and_rollback_generation(
        STALE_LEASE_GENERATION,
        lease["PublicationRunId"],
        workshop_generation_id(STALE_LEASE_GENERATION),
        previous_generation_id,
        "Recovering",
    )
    reason = (
        f"Explicit recovery rolled back stale generation {STALE_LEASE_GENERATION}; "
        "all final tables match the prior snapshots."
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("Generation") == STALE_LEASE_GENERATION)
            & (F.col("PublicationRunId") == lease["PublicationRunId"])
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == "Recovering")
        ),
        set={
            "OwnerRunId": F.lit(None).cast("string"),
            "PublishState": F.lit("Failed"),
            "LeaseExpiresAtUtc": F.lit(None).cast("timestamp"),
            "UpdatedAtUtc": F.lit(utc_now()),
            "FailureReason": F.lit(reason),
        },
    )
    recovered = current_lease_row()
    require(
        int(recovered["Generation"]) == STALE_LEASE_GENERATION
        and recovered["OwnerRunId"] is None
        and recovered["PublishState"] == "Failed",
        "Recovered generation was not safely released after rollback.",
    )


def acquire_publish_lease() -> int:
    lease = current_lease_row()
    owner = lease["OwnerRunId"]
    if owner is not None:
        expiry = lease["LeaseExpiresAtUtc"]
        if (
            lease["PublishState"] in ACTIVE_PUBLISH_STATES
            and expiry is not None
            and expiry <= utc_now()
        ):
            mark_expired_lease_stale(lease)
            lease = current_lease_row()
        if lease["PublishState"] == "Stale":
            if not OPERATOR_RECOVER_STALE_LEASE:
                raise RuntimeError(
                    "A stale publication lease exists. Explicit operator acknowledgement "
                    "is required; silent takeover is forbidden. After confirming that the "
                    "owning Spark session has stopped, rerun this notebook with "
                    "OPERATOR_RECOVER_STALE_LEASE = True, "
                    f"STALE_LEASE_OWNER_RUN_ID = \"{lease['OwnerRunId']}\", "
                    f"STALE_LEASE_GENERATION = {lease['Generation']}, "
                    "and restore both parameters to their defaults afterwards."
                )
            recover_stale_lease_explicitly(lease)
            lease = current_lease_row()
        else:
            raise RuntimeError(
                f"Active publication lease is owned by {owner} for generation "
                f"{lease['Generation']}; this run will not mutate final tables. "
                f"PublishState={lease['PublishState']}; "
                f"LeaseExpiresAtUtc={expiry}; UtcNow={utc_now()}. "
                "If that Spark session has already stopped, wait until "
                "LeaseExpiresAtUtc has passed and rerun this notebook; the run will "
                "then report the stale lease and the exact recovery parameters."
            )
    require(lease["OwnerRunId"] is None, "Lease owner was not safely cleared.")
    require(
        lease["PublishState"] in ("Ready", "Failed"),
        f"Lease state {lease['PublishState']} requires operator review before acquisition.",
    )
    expected_generation = int(lease["Generation"])
    generation = expected_generation + 1
    now = utc_now()
    expiry = now + timedelta(minutes=PUBLISH_LEASE_MINUTES)
    try:
        control_delta().update(
            condition=(
                (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
                & (F.col("RecordType") == "Lease")
                & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
                & (F.col("Generation") == expected_generation)
                & F.col("OwnerRunId").isNull()
                & (F.col("PublishState") == lease["PublishState"])
            ),
            set={
                "Generation": F.lit(generation).cast("long"),
                "PublicationRunId": F.lit(RUN_ID),
                "OwnerRunId": F.lit(RUN_ID),
                "PublishState": F.lit("Preparing"),
                "LeaseAcquiredAtUtc": F.lit(now),
                "LeaseExpiresAtUtc": F.lit(expiry),
                "PrePublishVersion": F.lit(-1).cast("long"),
                "ExpectedCurrentVersion": F.lit(-1).cast("long"),
                "PublishedVersion": F.lit(-1).cast("long"),
                "RollbackVersion": F.lit(-1).cast("long"),
                "UpdatedAtUtc": F.lit(now),
                "FailureReason": F.lit(None).cast("string"),
            },
        )
    except Exception as exc:
        raise RuntimeError("Concurrent publication lease acquisition was rejected by Delta.") from exc
    acquired = current_lease_row()
    require(
        int(acquired["Generation"]) == generation
        and acquired["PublicationRunId"] == RUN_ID
        and acquired["OwnerRunId"] == RUN_ID
        and acquired["PublishState"] == "Preparing",
        "Publication lease compare-before-update failed; another run won the fence.",
    )
    return generation


def assert_publish_lease(expected_states: tuple[str, ...]):
    lease = current_lease_row()
    require(
        int(lease["Generation"]) == PUBLISH_GENERATION,
        "Publication fencing generation changed.",
    )
    require(lease["PublicationRunId"] == RUN_ID, "Publication run generation changed.")
    require(lease["OwnerRunId"] == RUN_ID, "Publication lease ownership was lost.")
    require(lease["PublishState"] in expected_states, "Publication state changed unexpectedly.")
    expiry = lease["LeaseExpiresAtUtc"]
    if expiry is None or expiry <= utc_now():
        if lease["PublishState"] in ACTIVE_PUBLISH_STATES and expiry is not None:
            mark_expired_lease_stale(lease)
        raise RuntimeError("Publication lease expired; no further final-table mutation is allowed.")
    return lease


def renew_publish_lease(expected_states: tuple[str, ...]) -> None:
    lease = assert_publish_lease(expected_states)
    now = utc_now()
    new_expiry = now + timedelta(minutes=PUBLISH_LEASE_MINUTES)
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Lease")
            & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
            & (F.col("Generation") == PUBLISH_GENERATION)
            & (F.col("PublicationRunId") == RUN_ID)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == lease["PublishState"])
            & (F.col("LeaseExpiresAtUtc") > F.lit(now))
        ),
        set={
            "LeaseExpiresAtUtc": F.lit(new_expiry),
            "UpdatedAtUtc": F.lit(now),
        },
    )
    renewed = current_lease_row()
    require(
        renewed["OwnerRunId"] == RUN_ID
        and int(renewed["Generation"]) == PUBLISH_GENERATION
        and renewed["LeaseExpiresAtUtc"] == new_expiry,
        "Publication lease renewal lost a compare-before-update race.",
    )


def latest_delta_version(table_name: str) -> int:
    row = (
        spark.sql(f"DESCRIBE HISTORY {quoted_table(table_name)}")
        .select("version")
        .orderBy(F.desc("version"))
        .first()
    )
    require(row is not None, f"{table_name}: Delta history is unavailable.")
    return int(row["version"])


def current_delta_version(table_name: str) -> int:
    return latest_delta_version(table_name) if table_exists(table_name) else -1


FINAL_PRIMARY_KEYS = {
    **STAGE_PRIMARY_KEYS,
    **OUTPUT_PRIMARY_KEYS,
    AUDIT_TABLE: ["RunId", "SourceFile"],
}
FINAL_PUBLIC_COLUMNS = {
    **{
        table_name: [name for name, _ in schema]
        for table_name, schema in STAGING_SCHEMA_CONTRACT.items()
    },
    **{
        table_name: [name for name, _ in schema]
        for table_name, schema in OUTPUT_SCHEMA_CONTRACT.items()
    },
    AUDIT_TABLE: [name for name, _ in AUDIT_SCHEMA_CONTRACT],
}
NO_READY_GENERATION_ID = "furusato-v2.7.0-no-ready-generation"


def workshop_generation_id(generation: int) -> str:
    return f"furusato-v2.7.0-generation-{generation:020d}"


def quoted_column(name: str) -> str:
    require(bool(TABLE_NAME_PATTERN.fullmatch(name)), f"Unsafe column name: {name}")
    return f"`{name}`"


def sql_string_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def cas_failure_expression(table_name: str, expected_generation: str | None) -> str:
    expected_label = expected_generation if expected_generation is not None else "<no prior rows>"
    message = (
        f"{table_name}: atomic generation CAS rejected a target row outside "
        f"expected generation {expected_label}"
    )
    return f"CAST(raise_error({sql_string_literal(message)}) AS STRING)"


def replace_final_table_cas(
    table_name: str,
    source_df: DataFrame,
    expected_target_df: DataFrame,
    primary_keys: list[str],
    public_columns: list[str],
    expected_target_generation: str | None,
    replacement_generation: str,
) -> None:
    require(primary_keys, f"{table_name}: a documented primary key is required.")
    require(
        set(primary_keys).issubset(public_columns),
        f"{table_name}: primary key is outside the public schema.",
    )
    replacement_source = source_df.select(
        *[F.col(column).alias(column) for column in primary_keys],
        *[
            F.col(column).alias(f"__replacement_{column}")
            for column in public_columns
        ],
        F.lit(True).alias("__replacement_exists"),
    )
    expected_source = expected_target_df.select(
        *[F.col(column).alias(column) for column in primary_keys],
        *[
            F.col(column).alias(f"__expected_{column}")
            for column in public_columns
        ],
        F.lit(True).alias("__expected_exists"),
    )
    assert_unique(replacement_source, primary_keys, f"{table_name} replacement source PK")
    assert_unique(expected_source, primary_keys, f"{table_name} expected snapshot PK")
    merge_source = (
        replacement_source.join(expected_source, primary_keys, "full_outer")
        .fillna(False, subset=["__replacement_exists", "__expected_exists"])
    )
    match_condition = " AND ".join(
        f"target.{quoted_column(column)} <=> source.{quoted_column(column)}"
        for column in primary_keys
    )
    if expected_target_generation is None:
        expected_condition = "FALSE"
        unexpected_condition = "TRUE"
    else:
        expected_literal = sql_string_literal(expected_target_generation)
        target_generation = f"target.{quoted_column(WORKSHOP_GENERATION_COLUMN)}"
        expected_condition = f"{target_generation} <=> {expected_literal}"
    expected_row_matches = " AND ".join(
        [
            "source.`__expected_exists`",
            expected_condition,
            *[
                f"target.{quoted_column(column)} "
                f"<=> source.{quoted_column(f'__expected_{column}')}"
                for column in public_columns
            ],
        ]
    )
    failure_expression = cas_failure_expression(table_name, expected_target_generation)
    update_values = {
        column: f"source.{quoted_column(f'__replacement_{column}')}"
        for column in public_columns
    }
    update_values[WORKSHOP_GENERATION_COLUMN] = sql_string_literal(replacement_generation)
    insert_values = {
        column: f"source.{quoted_column(f'__replacement_{column}')}"
        for column in public_columns
    }
    insert_values[WORKSHOP_GENERATION_COLUMN] = sql_string_literal(replacement_generation)
    failure_values = dict(insert_values)
    failure_values[WORKSHOP_GENERATION_COLUMN] = failure_expression
    (
        DeltaTable.forName(spark, table_name)
        .alias("target")
        .merge(merge_source.alias("source"), match_condition)
        .whenMatchedUpdate(
            condition=f"({expected_row_matches}) AND source.`__replacement_exists`",
            set=update_values,
        )
        .whenMatchedDelete(
            condition=f"({expected_row_matches}) AND NOT source.`__replacement_exists`"
        )
        .whenMatchedUpdate(
            condition=f"NOT ({expected_row_matches})",
            set={WORKSHOP_GENERATION_COLUMN: failure_expression},
        )
        .whenNotMatchedInsert(
            condition="NOT source.`__expected_exists` AND source.`__replacement_exists`",
            values=insert_values,
        )
        .whenNotMatchedInsert(
            condition="source.`__expected_exists` OR NOT source.`__replacement_exists`",
            values=failure_values,
        )
        .whenNotMatchedBySourceUpdate(
            condition="TRUE",
            set={WORKSHOP_GENERATION_COLUMN: failure_expression},
        )
        .execute()
    )


def public_final_frame(table_name: str) -> DataFrame:
    return spark.table(table_name).select(*FINAL_PUBLIC_COLUMNS[table_name])


def assert_final_generation(table_name: str, expected_generation: str) -> None:
    target = spark.table(table_name)
    require(
        WORKSHOP_GENERATION_COLUMN in target.columns,
        f"{table_name}: technical generation column is missing.",
    )
    invalid = target.where(
        F.col(WORKSHOP_GENERATION_COLUMN).isNull()
        | (F.col(WORKSHOP_GENERATION_COLUMN) != expected_generation)
    )
    require(
        invalid.limit(1).count() == 0,
        f"{table_name}: rows do not share Ready generation {expected_generation}.",
    )


def latest_ready_generation(before_generation: int | None = None) -> int | None:
    rows = (
        control_rows()
        .where(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("PublishState") == "Ready")
        )
        .select("Generation", "PublicationRunId", "TableName")
        .collect()
    )
    grouped = {}
    for row in rows:
        generation = int(row["Generation"])
        if before_generation is not None and generation >= before_generation:
            continue
        grouped.setdefault(generation, []).append(row)
    valid = []
    expected_tables = set(FINAL_TABLE_ORDER)
    for generation, records in grouped.items():
        if (
            {row["TableName"] for row in records} == expected_tables
            and len({row["PublicationRunId"] for row in records}) == 1
        ):
            valid.append(generation)
    return max(valid) if valid else None


def prepublication_snapshot_is_unchanged(
    current_version: int,
    expected_version: int,
    matches_expected_content: bool,
) -> bool:
    return current_version == expected_version and matches_expected_content


def advance_prepublication_versions(
    expected_versions: dict[str, int],
    failed_evidence_rows: list,
) -> dict[str, int]:
    if not failed_evidence_rows:
        return dict(expected_versions)
    expected_tables = set(FINAL_TABLE_ORDER)
    require(
        len(failed_evidence_rows) == len(expected_tables)
        and {row["TableName"] for row in failed_evidence_rows} == expected_tables,
        "Failed-generation evidence is incomplete; pre-publication state cannot be inferred.",
    )
    require(
        len({row["PublicationRunId"] for row in failed_evidence_rows}) == 1,
        "Failed-generation evidence mixes publication runs.",
    )
    advanced = dict(expected_versions)
    for row in failed_evidence_rows:
        table_name = row["TableName"]
        pre_publish_version = int(row["PrePublishVersion"])
        expected_current_version = int(row["ExpectedCurrentVersion"])
        published_version = int(row["PublishedVersion"])
        rollback_version = int(row["RollbackVersion"])
        require(
            row["OwnerRunId"] is None and row["PublishState"] == "Failed",
            f"{table_name}: intervening generation is not immutably Failed.",
        )
        require(
            pre_publish_version == advanced[table_name],
            f"{table_name}: failed-generation evidence is not chained to the prior snapshot.",
        )
        if row["MutationState"] in ("Pending", "Intent"):
            require(
                expected_current_version == pre_publish_version
                and published_version == -1
                and rollback_version == -1,
                f"{table_name}: uncommitted failed evidence changed the expected snapshot.",
            )
        else:
            require(
                row["MutationState"] == "RolledBack"
                and published_version > pre_publish_version
                and rollback_version > published_version
                and expected_current_version == rollback_version,
                f"{table_name}: failed generation was not completely rolled back.",
            )
        advanced[table_name] = expected_current_version
    return advanced


def prepublication_snapshot_versions(
    previous_ready_generation: int | None,
    current_generation: int,
) -> tuple[dict[str, int], dict[str, int]]:
    if previous_ready_generation is None:
        return {}, {}
    ready_rows = (
        generation_rows(previous_ready_generation)
        .where(F.col("RecordType") == "Table")
        .collect()
    )
    expected_tables = set(FINAL_TABLE_ORDER)
    require(
        len(ready_rows) == len(expected_tables)
        and {row["TableName"] for row in ready_rows} == expected_tables,
        "Previous Ready generation evidence is incomplete.",
    )
    require(
        len({row["PublicationRunId"] for row in ready_rows}) == 1,
        "Previous Ready generation evidence mixes publication runs.",
    )
    ready_versions = {}
    for row in ready_rows:
        table_name = row["TableName"]
        published_version = int(row["PublishedVersion"])
        require(
            int(row["Generation"]) == previous_ready_generation
            and row["OwnerRunId"] is None
            and row["PublishState"] == "Ready"
            and row["MutationState"] == "Committed"
            and published_version >= 0
            and int(row["ExpectedCurrentVersion"]) == published_version,
            f"{table_name}: previous Ready snapshot evidence is inconsistent.",
        )
        ready_versions[table_name] = published_version
    expected_versions = dict(ready_versions)
    for generation in range(previous_ready_generation + 1, current_generation):
        failed_rows = (
            generation_rows(generation)
            .where(F.col("RecordType") == "Table")
            .collect()
        )
        expected_versions = advance_prepublication_versions(expected_versions, failed_rows)
    return ready_versions, expected_versions


def ensure_final_merge_target(
    table_name: str,
    candidate_df: DataFrame,
    previous_ready_generation: int | None,
    expected_prepublication_version: int | None,
    ready_snapshot_version: int | None,
    current_generation_id: str,
) -> int:
    expected_columns = FINAL_PUBLIC_COLUMNS[table_name] + [WORKSHOP_GENERATION_COLUMN]
    if table_exists(table_name):
        target = spark.table(table_name)
        require(
            target.columns == expected_columns,
            f"{table_name}: existing table is unfenced or has an incompatible schema. "
            "Automatic overwrite is forbidden.",
        )
        if previous_ready_generation is None:
            require(
                expected_prepublication_version is None and ready_snapshot_version is None,
                f"{table_name}: bootstrap publication received unexpected Ready evidence.",
            )
            require(
                target.limit(1).count() == 0,
                f"{table_name}: non-empty final data has no immutable Ready evidence.",
            )
            return current_delta_version(table_name)
        require(
            expected_prepublication_version is not None and ready_snapshot_version is not None,
            f"{table_name}: previous Ready snapshot versions are missing.",
        )
        current_version = current_delta_version(table_name)
        current_rows = target.select(*FINAL_PUBLIC_COLUMNS[table_name])
        expected_rows = version_as_of_public(table_name, expected_prepublication_version)
        ready_rows = version_as_of_public(table_name, ready_snapshot_version)
        require(
            frames_match_exact(
                expected_rows,
                ready_rows,
                FINAL_PUBLIC_COLUMNS[table_name],
            ),
            f"{table_name}: rollback evidence is not anchored to the previous Ready content.",
        )
        require(
            prepublication_snapshot_is_unchanged(
                current_version,
                expected_prepublication_version,
                frames_match_exact(
                    current_rows,
                    expected_rows,
                    FINAL_PUBLIC_COLUMNS[table_name],
                ),
            ),
            f"{table_name}: pre-publication version/content drift was detected.",
        )
        assert_final_generation(
            table_name,
            workshop_generation_id(previous_ready_generation),
        )
        return expected_prepublication_version
    require(
        previous_ready_generation is None
        and expected_prepublication_version is None
        and ready_snapshot_version is None,
        f"{table_name}: a table from the previous Ready generation is missing.",
    )
    assert_publish_lease(("Preparing",))
    (
        candidate_df.select(*FINAL_PUBLIC_COLUMNS[table_name])
        .withColumn(
            WORKSHOP_GENERATION_COLUMN,
            F.lit(current_generation_id).cast("string"),
        )
        .limit(0)
        .write.format("delta")
        .mode("errorifexists")
        .saveAsTable(table_name)
    )
    return current_delta_version(table_name)


def initialize_generation_evidence(prior_versions: dict[str, int]) -> None:
    assert_publish_lease(("Preparing",))
    now = utc_now()
    evidence_rows = [
        (
            PUBLISH_CONTROL_KEY,
            "Table",
            table_name,
            PUBLISH_GENERATION,
            RUN_ID,
            RUN_ID,
            "Preparing",
            None,
            None,
            int(prior_versions[table_name]),
            int(prior_versions[table_name]),
            -1,
            -1,
            TEMP_TABLE_MAP[table_name],
            "Pending",
            now,
            None,
        )
        for table_name in FINAL_TABLE_ORDER
    ]
    evidence = spark.createDataFrame(evidence_rows, schema=PUBLISH_CONTROL_SCHEMA)
    existing = generation_rows(PUBLISH_GENERATION).where(F.col("RecordType") == "Table").count()
    require(existing == 0, "Generation evidence already exists; generation reuse is forbidden.")
    (
        control_delta()
        .alias("target")
        .merge(
            evidence.alias("source"),
            "target.ControlKey = source.ControlKey "
            "AND target.RecordType = source.RecordType "
            "AND target.Generation = source.Generation "
            "AND target.TableName = source.TableName",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    records = generation_rows(PUBLISH_GENERATION)
    require(records.count() == EXPECTED_GENERATION_RECORDS, "Incomplete Preparing evidence set.")
    assert_unique(
        records,
        ["ControlKey", "RecordType", "Generation", "TableName"],
        "publish generation evidence",
    )


def generation_evidence_row(table_name: str, generation: int | None = None):
    selected_generation = PUBLISH_GENERATION if generation is None else generation
    rows = (
        generation_rows(selected_generation)
        .where((F.col("RecordType") == "Table") & (F.col("TableName") == table_name))
        .collect()
    )
    require(len(rows) == 1, f"{table_name}: generation evidence row is missing or duplicated.")
    return rows[0]


def transition_generation_state(expected_state: str, new_state: str, release: bool) -> None:
    assert_publish_lease((expected_state,))
    records = generation_rows(PUBLISH_GENERATION).collect()
    require(len(records) == EXPECTED_GENERATION_RECORDS, "Generation evidence is incomplete.")
    require(
        all(
            row["PublicationRunId"] == RUN_ID
            and row["OwnerRunId"] == RUN_ID
            and row["PublishState"] == expected_state
            for row in records
        ),
        f"Generation records are not uniformly {expected_state}.",
    )
    now = utc_now()
    updates = {
        "PublishState": F.lit(new_state),
        "UpdatedAtUtc": F.lit(now),
        "FailureReason": F.lit(None).cast("string"),
    }
    if release:
        updates["OwnerRunId"] = F.lit(None).cast("string")
        updates["LeaseExpiresAtUtc"] = F.lit(None).cast("timestamp")
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("Generation") == PUBLISH_GENERATION)
            & (F.col("PublicationRunId") == RUN_ID)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == expected_state)
        ),
        set=updates,
    )
    changed = generation_rows(PUBLISH_GENERATION).collect()
    if release and new_state == "Ready":
        table_records = [row for row in changed if row["RecordType"] == "Table"]
        require(
            len(table_records) == len(FINAL_TABLE_ORDER),
            "Ready transition lost immutable table evidence rows.",
        )
        changed = table_records
    else:
        require(len(changed) == EXPECTED_GENERATION_RECORDS, "State transition lost evidence rows.")
    require(
        all(
            row["PublishState"] == new_state
            and row["PublicationRunId"] == RUN_ID
            and (row["OwnerRunId"] is None if release else row["OwnerRunId"] == RUN_ID)
            for row in changed
        ),
        f"Generation state transition to {new_state} was not atomic in the control table.",
    )


def record_mutation_intent(table_name: str) -> None:
    assert_publish_lease(("Publishing",))
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("TableName") == table_name)
            & (F.col("Generation") == PUBLISH_GENERATION)
            & (F.col("PublicationRunId") == RUN_ID)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == "Publishing")
            & (F.col("MutationState") == "Pending")
        ),
        set={
            "MutationState": F.lit("Intent"),
            "UpdatedAtUtc": F.lit(utc_now()),
        },
    )
    require(
        generation_evidence_row(table_name)["MutationState"] == "Intent",
        f"{table_name}: mutation intent was not durably recorded.",
    )


def record_published_version(
    table_name: str,
    prior_version: int,
    candidate_rows: DataFrame,
    replacement_generation_id: str,
    allow_superseded_current: bool = False,
    generation: int | None = None,
    publication_run_id: str | None = None,
    publish_state: str = "Publishing",
) -> None:
    selected_generation = PUBLISH_GENERATION if generation is None else generation
    selected_run_id = RUN_ID if publication_run_id is None else publication_run_id
    expected_version = verify_expected_publication_commit(
        table_name,
        prior_version,
        candidate_rows,
        replacement_generation_id,
        allow_superseded_current,
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("TableName") == table_name)
            & (F.col("Generation") == selected_generation)
            & (F.col("PublicationRunId") == selected_run_id)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == publish_state)
            & (F.col("ExpectedCurrentVersion") == prior_version)
            & (F.col("MutationState") == "Intent")
        ),
        set={
            "ExpectedCurrentVersion": F.lit(expected_version).cast("long"),
            "PublishedVersion": F.lit(expected_version).cast("long"),
            "MutationState": F.lit("Committed"),
            "UpdatedAtUtc": F.lit(utc_now()),
        },
    )
    evidence = generation_evidence_row(table_name, selected_generation)
    require(
        int(evidence["ExpectedCurrentVersion"]) == expected_version
        and int(evidence["PublishedVersion"]) == expected_version
        and evidence["MutationState"] == "Committed",
        f"{table_name}: published version evidence update failed.",
    )
    verify_expected_publication_commit(
        table_name,
        prior_version,
        candidate_rows,
        replacement_generation_id,
        allow_superseded_current,
    )


def record_rollback_intent(
    table_name: str,
    expected_published_version: int,
    observed_pre_rollback_version: int,
    generation: int,
    publication_run_id: str,
    publish_state: str,
) -> None:
    require(
        observed_pre_rollback_version >= expected_published_version,
        f"{table_name}: rollback observation predates the publication commit.",
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("TableName") == table_name)
            & (F.col("Generation") == generation)
            & (F.col("PublicationRunId") == publication_run_id)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == publish_state)
            & (F.col("ExpectedCurrentVersion") == expected_published_version)
            & (F.col("PublishedVersion") == expected_published_version)
            & (F.col("MutationState") == "Committed")
        ),
        set={
            "ExpectedCurrentVersion": F.lit(observed_pre_rollback_version).cast("long"),
            "MutationState": F.lit("RollbackIntent"),
            "UpdatedAtUtc": F.lit(utc_now()),
        },
    )
    evidence = generation_evidence_row(table_name, generation)
    require(
        evidence["MutationState"] == "RollbackIntent"
        and int(evidence["ExpectedCurrentVersion"]) == observed_pre_rollback_version,
        f"{table_name}: rollback intent/version anchor was not durably recorded.",
    )


def record_rollback_version(
    table_name: str,
    observed_pre_rollback_version: int,
    previous_generation_id: str,
    pre_publish_version: int,
    generation: int,
    publication_run_id: str,
    publish_state: str,
) -> None:
    expected_rollback_version = verify_expected_rollback_commit(
        table_name,
        observed_pre_rollback_version,
        pre_publish_version,
        previous_generation_id,
    )
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Table")
            & (F.col("TableName") == table_name)
            & (F.col("Generation") == generation)
            & (F.col("PublicationRunId") == publication_run_id)
            & (F.col("OwnerRunId") == RUN_ID)
            & (F.col("PublishState") == publish_state)
            & (F.col("ExpectedCurrentVersion") == observed_pre_rollback_version)
            & (F.col("MutationState") == "RollbackIntent")
        ),
        set={
            "ExpectedCurrentVersion": F.lit(expected_rollback_version).cast("long"),
            "RollbackVersion": F.lit(expected_rollback_version).cast("long"),
            "MutationState": F.lit("RolledBack"),
            "UpdatedAtUtc": F.lit(utc_now()),
        },
    )
    evidence = generation_evidence_row(table_name, generation)
    require(
        int(evidence["ExpectedCurrentVersion"]) == expected_rollback_version
        and int(evidence["RollbackVersion"]) == expected_rollback_version
        and evidence["MutationState"] == "RolledBack",
        f"{table_name}: rollback completion evidence update failed.",
    )
    verify_expected_rollback_commit(
        table_name,
        observed_pre_rollback_version,
        pre_publish_version,
        previous_generation_id,
    )


def control_generation_is_ready(
    lease,
    evidence_rows: list,
    generation: int,
    publication_run_id: str,
    expected_tables: set[str],
) -> bool:
    lease_generation = int(lease["Generation"])
    completed_in_lease = (
        lease_generation == generation
        and lease["PublicationRunId"] == publication_run_id
        and lease["OwnerRunId"] is None
        and lease["PublishState"] == "Ready"
    )
    advanced_after_completion = lease_generation > generation
    if not (completed_in_lease or advanced_after_completion):
        return False
    if len(evidence_rows) != len(expected_tables):
        return False
    return (
        {row["TableName"] for row in evidence_rows} == expected_tables
        and all(
            int(row["Generation"]) == generation
            and row["PublicationRunId"] == publication_run_id
            and row["OwnerRunId"] is None
            and row["PublishState"] == "Ready"
            for row in evidence_rows
        )
    )


def verify_generation_tables(expected_state: str) -> bool:
    records = generation_rows(PUBLISH_GENERATION).collect()
    table_records = [row for row in records if row["RecordType"] == "Table"]
    lease = current_lease_row()
    if expected_state == "Ready":
        require(
            control_generation_is_ready(
                lease,
                table_records,
                PUBLISH_GENERATION,
                RUN_ID,
                set(FINAL_TABLE_ORDER),
            ),
            "Ready control/evidence generation is inconsistent.",
        )
        if int(lease["Generation"]) > PUBLISH_GENERATION:
            return True
    else:
        require(
            len(records) == EXPECTED_GENERATION_RECORDS,
            "Active generation evidence is incomplete.",
        )
        require(
            all(
                int(row["Generation"]) == PUBLISH_GENERATION
                and row["PublicationRunId"] == RUN_ID
                and row["OwnerRunId"] == RUN_ID
                and row["PublishState"] == expected_state
                for row in records
            ),
            f"Generation evidence is not uniformly {expected_state}.",
        )
    expected_generation_id = workshop_generation_id(PUBLISH_GENERATION)
    for evidence in table_records:
        table_name = evidence["TableName"]
        expected_version = int(evidence["ExpectedCurrentVersion"])
        require(
            int(evidence["PublishedVersion"]) == expected_version,
            f"{table_name}: published/control versions differ.",
        )
        require(
            evidence["MutationState"] == "Committed",
            f"{table_name}: publication intent is not durably committed.",
        )
        require(
            current_delta_version(table_name) == expected_version,
            f"{table_name}: Delta version no longer matches control evidence.",
        )
        assert_final_generation(table_name, expected_generation_id)
    return False


def validate_ready_generation() -> None:
    if verify_generation_tables("Ready"):
        return
    validate_stage_frames(
        {name: public_final_frame(name) for name in STAGE_TABLE_ORDER}
    )
    validate_output_frames(
        {name: public_final_frame(name) for name in OUTPUT_TABLE_ORDER}
    )
    final_audit = public_final_frame(AUDIT_TABLE)
    assert_schema(final_audit, AUDIT_SCHEMA_CONTRACT, AUDIT_TABLE)
    assert_row_count(final_audit, 8, AUDIT_TABLE)


def fail_owned_generation(reason: str) -> bool:
    lease = current_lease_row()
    if not (
        int(lease["Generation"]) == PUBLISH_GENERATION
        and lease["PublicationRunId"] == RUN_ID
        and lease["OwnerRunId"] == RUN_ID
        and lease["PublishState"] in ACTIVE_PUBLISH_STATES
    ):
        return False
    now = utc_now()
    control_delta().update(
        condition=(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("Generation") == PUBLISH_GENERATION)
            & (F.col("PublicationRunId") == RUN_ID)
            & (F.col("OwnerRunId") == RUN_ID)
            & F.col("PublishState").isin(*ACTIVE_PUBLISH_STATES)
        ),
        set={
            "OwnerRunId": F.lit(None).cast("string"),
            "PublishState": F.lit("Failed"),
            "LeaseExpiresAtUtc": F.lit(None).cast("timestamp"),
            "UpdatedAtUtc": F.lit(now),
            "FailureReason": F.lit(reason[:4000]),
        },
    )
    failed = current_lease_row()
    return (
        int(failed["Generation"]) == PUBLISH_GENERATION
        and failed["OwnerRunId"] is None
        and failed["PublishState"] == "Failed"
    )


def classify_ready_failure_action(lease, generation: int, publication_run_id: str) -> str:
    lease_generation = int(lease["Generation"])
    if lease_generation > generation:
        return "completed"
    if (
        lease_generation == generation
        and lease["PublicationRunId"] == publication_run_id
        and lease["OwnerRunId"] is None
        and lease["PublishState"] == "Ready"
    ):
        return "recover"
    return "invalid"


def take_ready_recovery_ownership(reason: str) -> bool:
    lease = current_lease_row()
    action = classify_ready_failure_action(lease, PUBLISH_GENERATION, RUN_ID)
    if action == "completed":
        table_records = (
            generation_rows(PUBLISH_GENERATION)
            .where(F.col("RecordType") == "Table")
            .collect()
        )
        require(
            control_generation_is_ready(
                lease,
                table_records,
                PUBLISH_GENERATION,
                RUN_ID,
                set(FINAL_TABLE_ORDER),
            ),
            "Newer lease advanced without immutable historical Ready evidence.",
        )
        return False
    require(action == "recover", "Ready generation is not eligible for owned recovery.")
    now = utc_now()
    expiry = now + timedelta(minutes=PUBLISH_LEASE_MINUTES)
    ready_gate = (
        control_rows()
        .where(
            (F.col("ControlKey") == PUBLISH_CONTROL_KEY)
            & (F.col("RecordType") == "Lease")
            & (F.col("TableName") == PUBLISH_CONTROL_RECORD)
            & (F.col("Generation") == PUBLISH_GENERATION)
            & (F.col("PublicationRunId") == RUN_ID)
            & F.col("OwnerRunId").isNull()
            & (F.col("PublishState") == "Ready")
        )
        .select(F.lit(True).alias("__ready_gate"))
        .limit(1)
    )
    recovery_source = generation_rows(PUBLISH_GENERATION).crossJoin(ready_gate).select(
        "ControlKey",
        "RecordType",
        "TableName",
        "Generation",
        "PublicationRunId",
    )
    (
        control_delta()
        .alias("target")
        .merge(
            recovery_source.alias("source"),
            "target.ControlKey = source.ControlKey "
            "AND target.RecordType = source.RecordType "
            "AND target.TableName = source.TableName "
            "AND target.Generation = source.Generation "
            "AND target.PublicationRunId = source.PublicationRunId",
        )
        .whenMatchedUpdate(
            condition="target.OwnerRunId IS NULL AND target.PublishState = 'Ready'",
            set={
                "OwnerRunId": F.lit(RUN_ID),
                "PublishState": F.lit("Recovering"),
                "LeaseExpiresAtUtc": F.lit(expiry),
                "UpdatedAtUtc": F.lit(now),
                "FailureReason": F.lit(reason[:4000]),
            },
        )
        .execute()
    )
    recovered_lease = current_lease_row()
    post_action = classify_ready_failure_action(
        recovered_lease,
        PUBLISH_GENERATION,
        RUN_ID,
    )
    if post_action == "completed":
        historical_records = (
            generation_rows(PUBLISH_GENERATION)
            .where(F.col("RecordType") == "Table")
            .collect()
        )
        require(
            control_generation_is_ready(
                recovered_lease,
                historical_records,
                PUBLISH_GENERATION,
                RUN_ID,
                set(FINAL_TABLE_ORDER),
            ),
            "Ready recovery lost a race and altered historical evidence.",
        )
        return False
    recovered_records = generation_rows(PUBLISH_GENERATION).collect()
    require(
        len(recovered_records) == EXPECTED_GENERATION_RECORDS
        and all(
            row["PublicationRunId"] == RUN_ID
            and row["OwnerRunId"] == RUN_ID
            and row["PublishState"] == "Recovering"
            for row in recovered_records
        ),
        "Ready generation was not atomically taken into owned recovery.",
    )
    return True


def recover_ready_generation(reason: str) -> bool:
    if not take_ready_recovery_ownership(reason):
        return False
    renew_publish_lease(("Recovering",))
    rolled_back = reconcile_and_rollback_generation(
        PUBLISH_GENERATION,
        RUN_ID,
        CURRENT_GENERATION_ID,
        PREVIOUS_GENERATION_ID,
        "Recovering",
    )
    require(
        len(rolled_back) == len(FINAL_TABLE_ORDER)
        and set(rolled_back) == set(FINAL_TABLE_ORDER),
        "Ready verification recovery did not roll back every published table.",
    )
    require(
        fail_owned_generation(
            f"{reason[:3500]} Complete rollback restored the previous Ready generation."
        ),
        "Recovered Ready generation was not safely marked Failed after rollback.",
    )
    return True


def version_as_of_public(table_name: str, version: int) -> DataFrame:
    return (
        spark.read.format("delta")
        .option("versionAsOf", version)
        .table(table_name)
        .select(*FINAL_PUBLIC_COLUMNS[table_name])
    )


def rollback_table_cas(
    table_name: str,
    failed_generation_id: str,
    previous_generation_id: str,
    pre_publish_version: int,
    observed_pre_rollback_version: int,
    expected_failed_rows: DataFrame,
) -> int:
    previous_rows = version_as_of_public(table_name, pre_publish_version)
    replace_final_table_cas(
        table_name,
        previous_rows,
        expected_failed_rows,
        FINAL_PRIMARY_KEYS[table_name],
        FINAL_PUBLIC_COLUMNS[table_name],
        failed_generation_id,
        previous_generation_id,
    )
    return verify_expected_rollback_commit(
        table_name,
        observed_pre_rollback_version,
        pre_publish_version,
        previous_generation_id,
    )


def frames_match_exact(left: DataFrame, right: DataFrame, columns: list[str]) -> bool:
    selected_left = left.select(*columns)
    selected_right = right.select(*columns)
    return (
        selected_left.exceptAll(selected_right).limit(1).count() == 0
        and selected_right.exceptAll(selected_left).limit(1).count() == 0
    )


def frame_matches_candidate_generation(
    target: DataFrame,
    candidate_rows: DataFrame,
    public_columns: list[str],
    replacement_generation_id: str,
) -> bool:
    generation_drift = target.where(
        F.col(WORKSHOP_GENERATION_COLUMN).isNull()
        | (F.col(WORKSHOP_GENERATION_COLUMN) != replacement_generation_id)
    )
    return (
        generation_drift.limit(1).count() == 0
        and frames_match_exact(target, candidate_rows, public_columns)
    )


def version_matches_candidate_generation(
    table_name: str,
    version: int,
    candidate_rows: DataFrame,
    replacement_generation_id: str,
) -> bool:
    target = (
        spark.read.format("delta")
        .option("versionAsOf", version)
        .table(table_name)
    )
    return frame_matches_candidate_generation(
        target,
        candidate_rows,
        FINAL_PUBLIC_COLUMNS[table_name],
        replacement_generation_id,
    )


def classify_publication_commit_version(
    current_version: int,
    prior_version: int,
    matches_expected_snapshot: bool,
    matches_current_candidate: bool,
) -> str:
    if not (matches_expected_snapshot and matches_current_candidate):
        return "drift"
    expected_version = prior_version + 1
    if current_version == expected_version:
        return "exact"
    if current_version > expected_version:
        return "superseded"
    return "drift"


def verify_expected_publication_commit(
    table_name: str,
    prior_version: int,
    candidate_rows: DataFrame,
    replacement_generation_id: str,
    allow_superseded_current: bool = False,
) -> int:
    expected_version = prior_version + 1
    current_target = spark.table(table_name)
    commit_state = classify_publication_commit_version(
        current_delta_version(table_name),
        prior_version,
        version_matches_candidate_generation(
            table_name,
            expected_version,
            candidate_rows,
            replacement_generation_id,
        ),
        frame_matches_candidate_generation(
            current_target,
            candidate_rows,
            FINAL_PUBLIC_COLUMNS[table_name],
            replacement_generation_id,
        ),
    )
    allowed_states = ("exact", "superseded") if allow_superseded_current else ("exact",)
    require(
        commit_state in allowed_states,
        f"{table_name}: publication commit is not exactly version {expected_version} "
        "with candidate content and generation.",
    )
    return expected_version


def verify_expected_rollback_commit(
    table_name: str,
    observed_pre_rollback_version: int,
    pre_publish_version: int,
    previous_generation_id: str,
) -> int:
    expected_rollback_version = observed_pre_rollback_version + 1
    previous_rows = version_as_of_public(table_name, pre_publish_version)
    rollback_state = classify_publication_commit_version(
        current_delta_version(table_name),
        observed_pre_rollback_version,
        version_matches_candidate_generation(
            table_name,
            expected_rollback_version,
            previous_rows,
            previous_generation_id,
        ),
        frame_matches_candidate_generation(
            spark.table(table_name),
            previous_rows,
            FINAL_PUBLIC_COLUMNS[table_name],
            previous_generation_id,
        ),
    )
    require(
        rollback_state == "exact",
        f"{table_name}: rollback commit is not exactly version {expected_rollback_version} "
        "with previous snapshot content and generation.",
    )
    return expected_rollback_version


def classify_publication_observation(
    mutation_state: str,
    current_version: int,
    pre_publish_version: int,
    expected_current_version: int,
    generation_status: str,
    matches_candidate: bool,
    matches_previous: bool,
    matches_expected_commit: bool,
    matches_expected_rollback: bool,
) -> str:
    uncommitted = (
        current_version == pre_publish_version
        and generation_status in ("previous", "empty")
        and matches_previous
    )
    commit_state = classify_publication_commit_version(
        current_version,
        pre_publish_version,
        matches_expected_commit,
        generation_status in ("replacement", "empty") and matches_candidate,
    )
    committed = commit_state == "exact"
    superseded_commit = commit_state == "superseded"
    rollback_state = classify_publication_commit_version(
        current_version,
        expected_current_version,
        matches_expected_rollback,
        generation_status in ("previous", "empty") and matches_previous,
    )
    rolled_back = rollback_state == "exact"
    superseded_rollback = rollback_state == "superseded"
    if mutation_state == "Pending":
        return "uncommitted" if uncommitted else "drift"
    if mutation_state == "Intent":
        if uncommitted:
            return "uncommitted"
        if committed:
            return "committed"
        return "superseded_commit" if superseded_commit else "drift"
    if mutation_state == "Committed":
        if committed and current_version == expected_current_version:
            return "committed"
        return "superseded_commit" if superseded_commit else "drift"
    if mutation_state == "RollbackIntent":
        if committed and current_version == expected_current_version:
            return "committed"
        if superseded_commit and current_version == expected_current_version:
            return "superseded_commit"
        if rolled_back:
            return "rolled_back"
        return "superseded_rollback" if superseded_rollback else "drift"
    if mutation_state == "RolledBack":
        return (
            "rolled_back"
            if current_version == expected_current_version
            and generation_status in ("previous", "empty")
            and matches_previous
            else "drift"
        )
    return "drift"


def observe_table_publication(
    evidence,
    replacement_generation_id: str,
    previous_generation_id: str,
) -> dict:
    table_name = evidence["TableName"]
    candidate_table_name = evidence["CandidateTableName"]
    require(
        candidate_table_name and table_exists(candidate_table_name),
        f"{table_name}: retained candidate table is unavailable; recovery cannot prove state.",
    )
    public_columns = FINAL_PUBLIC_COLUMNS[table_name]
    target = spark.table(table_name)
    generation_values = [
        row[WORKSHOP_GENERATION_COLUMN]
        for row in target.select(WORKSHOP_GENERATION_COLUMN).distinct().limit(2).collect()
    ]
    if not generation_values:
        generation_status = "empty"
    elif generation_values == [replacement_generation_id]:
        generation_status = "replacement"
    elif generation_values == [previous_generation_id]:
        generation_status = "previous"
    else:
        generation_status = "other"
    current_rows = target.select(*public_columns)
    candidate_rows = spark.table(candidate_table_name).select(*public_columns)
    pre_publish_version = int(evidence["PrePublishVersion"])
    previous_rows = version_as_of_public(table_name, pre_publish_version)
    current_version = current_delta_version(table_name)
    matches_expected_commit = (
        version_matches_candidate_generation(
            table_name,
            pre_publish_version + 1,
            candidate_rows,
            replacement_generation_id,
        )
        if current_version > pre_publish_version
        else False
    )
    observed_rollback_version = int(evidence["ExpectedCurrentVersion"])
    matches_expected_rollback = (
        version_matches_candidate_generation(
            table_name,
            observed_rollback_version + 1,
            previous_rows,
            previous_generation_id,
        )
        if evidence["MutationState"] == "RollbackIntent"
        and current_version > observed_rollback_version
        else False
    )
    classification = classify_publication_observation(
        evidence["MutationState"],
        current_version,
        pre_publish_version,
        observed_rollback_version,
        generation_status,
        frames_match_exact(current_rows, candidate_rows, public_columns),
        frames_match_exact(current_rows, previous_rows, public_columns),
        matches_expected_commit,
        matches_expected_rollback,
    )
    return {
        "table_name": table_name,
        "evidence": evidence,
        "candidate_rows": candidate_rows,
        "current_version": current_version,
        "classification": classification,
    }


def reconcile_and_rollback_generation(
    generation: int,
    publication_run_id: str,
    replacement_generation_id: str,
    previous_generation_id: str,
    publish_state: str,
) -> list[str]:
    evidence_rows = (
        generation_rows(generation)
        .where(F.col("RecordType") == "Table")
        .collect()
    )
    require(
        len(evidence_rows) == len(FINAL_TABLE_ORDER)
        and {row["TableName"] for row in evidence_rows} == set(FINAL_TABLE_ORDER),
        "Generation table evidence is incomplete; recovery refuses to guess.",
    )
    observations = [
        observe_table_publication(
            evidence,
            replacement_generation_id,
            previous_generation_id,
        )
        for evidence in evidence_rows
    ]
    drift = [
        observation["table_name"]
        for observation in observations
        if observation["classification"] in ("drift", "superseded_rollback")
    ]
    require(
        not drift,
        f"Unexplained final-table drift detected; recovery ownership retained: {drift}",
    )
    rolled_back = []
    by_name = {observation["table_name"]: observation for observation in observations}
    for table_name in reversed(FINAL_TABLE_ORDER):
        observation = by_name[table_name]
        evidence = observation["evidence"]
        classification = observation["classification"]
        if classification == "uncommitted":
            continue
        if classification == "rolled_back":
            if evidence["MutationState"] == "RollbackIntent":
                record_rollback_version(
                    table_name,
                    int(evidence["ExpectedCurrentVersion"]),
                    previous_generation_id,
                    int(evidence["PrePublishVersion"]),
                    generation,
                    publication_run_id,
                    publish_state,
                )
            rolled_back.append(table_name)
            continue
        if evidence["MutationState"] == "Intent":
            record_published_version(
                table_name,
                int(evidence["PrePublishVersion"]),
                observation["candidate_rows"],
                replacement_generation_id,
                allow_superseded_current=True,
                generation=generation,
                publication_run_id=publication_run_id,
                publish_state=publish_state,
            )
            evidence = generation_evidence_row(table_name, generation)
        if evidence["MutationState"] == "RollbackIntent":
            expected_published_version = int(evidence["PublishedVersion"])
            observed_pre_rollback_version = int(evidence["ExpectedCurrentVersion"])
        else:
            expected_published_version = int(evidence["ExpectedCurrentVersion"])
            observed_pre_rollback_version = observation["current_version"]
            record_rollback_intent(
                table_name,
                expected_published_version,
                observed_pre_rollback_version,
                generation,
                publication_run_id,
                publish_state,
            )
        try:
            rollback_table_cas(
                table_name,
                replacement_generation_id,
                previous_generation_id,
                int(evidence["PrePublishVersion"]),
                observed_pre_rollback_version,
                observation["candidate_rows"],
            )
        except Exception:
            refreshed = observe_table_publication(
                generation_evidence_row(table_name, generation),
                replacement_generation_id,
                previous_generation_id,
            )
            if refreshed["classification"] != "rolled_back":
                raise
        record_rollback_version(
            table_name,
            observed_pre_rollback_version,
            previous_generation_id,
            int(evidence["PrePublishVersion"]),
            generation,
            publication_run_id,
            publish_state,
        )
        rolled_back.append(table_name)
    return rolled_back


ensure_publish_control_table()
PUBLISH_GENERATION = None
PREVIOUS_READY_GENERATION = None
CURRENT_GENERATION_ID = None
PREVIOUS_GENERATION_ID = None
ready_snapshot_versions = {}
expected_prior_versions = {}
prior_versions = {}
publish_succeeded = False
ready_failure_recovered = False
rollback_errors = []

try:
    PUBLISH_GENERATION = acquire_publish_lease()
    CURRENT_GENERATION_ID = workshop_generation_id(PUBLISH_GENERATION)
    PREVIOUS_READY_GENERATION = latest_ready_generation(PUBLISH_GENERATION)
    PREVIOUS_GENERATION_ID = (
        workshop_generation_id(PREVIOUS_READY_GENERATION)
        if PREVIOUS_READY_GENERATION is not None
        else NO_READY_GENERATION_ID
    )
    renew_publish_lease(("Preparing",))
    ready_snapshot_versions, expected_prior_versions = prepublication_snapshot_versions(
        PREVIOUS_READY_GENERATION,
        PUBLISH_GENERATION,
    )
    for final_name in FINAL_TABLE_ORDER:
        renew_publish_lease(("Preparing",))
        prior_versions[final_name] = ensure_final_merge_target(
            final_name,
            candidate_frames[final_name],
            PREVIOUS_READY_GENERATION,
            (
                expected_prior_versions[final_name]
                if PREVIOUS_READY_GENERATION is not None
                else None
            ),
            (
                ready_snapshot_versions[final_name]
                if PREVIOUS_READY_GENERATION is not None
                else None
            ),
            CURRENT_GENERATION_ID,
        )
    initialize_generation_evidence(prior_versions)
    transition_generation_state("Preparing", "Publishing", release=False)

    for final_name in FINAL_TABLE_ORDER:
        renew_publish_lease(("Publishing",))
        record_mutation_intent(final_name)
        replace_final_table_cas(
            final_name,
            candidate_frames[final_name],
            version_as_of_public(final_name, prior_versions[final_name]),
            FINAL_PRIMARY_KEYS[final_name],
            FINAL_PUBLIC_COLUMNS[final_name],
            (
                workshop_generation_id(PREVIOUS_READY_GENERATION)
                if PREVIOUS_READY_GENERATION is not None
                else None
            ),
            CURRENT_GENERATION_ID,
        )
        record_published_version(
            final_name,
            prior_versions[final_name],
            candidate_frames[final_name],
            CURRENT_GENERATION_ID,
        )

    validate_stage_frames(
        {name: public_final_frame(name) for name in STAGE_TABLE_ORDER}
    )
    validate_output_frames(
        {name: public_final_frame(name) for name in OUTPUT_TABLE_ORDER}
    )
    final_audit = public_final_frame(AUDIT_TABLE)
    assert_schema(final_audit, AUDIT_SCHEMA_CONTRACT, AUDIT_TABLE)
    assert_row_count(final_audit, 8, AUDIT_TABLE)
    verify_generation_tables("Publishing")

    if RUN_OPTIMIZE_VORDER:
        print(
            "RUN_OPTIMIZE_VORDER was requested, but optimization is deliberately "
            "deferred until after Ready and is not executed by this publication path."
        )

    transition_generation_state("Publishing", "Ready", release=True)
    try:
        validate_ready_generation()
    except Exception as ready_error:
        ready_failure_recovered = recover_ready_generation(
            f"Ready verification failed: {ready_error}"
        )
        if ready_failure_recovered:
            raise
        print(
            "Ready verification raced with a newer acquired generation; "
            "immutable historical Ready evidence was preserved."
        )
    publish_succeeded = True
except Exception as publish_error:
    rollback_completed = ready_failure_recovered
    if (
        not ready_failure_recovered
        and PUBLISH_GENERATION is not None
        and CURRENT_GENERATION_ID is not None
    ):
        try:
            lease = current_lease_row()
            if (
                int(lease["Generation"]) == PUBLISH_GENERATION
                and lease["PublicationRunId"] == RUN_ID
                and lease["OwnerRunId"] == RUN_ID
                and lease["PublishState"] == "Publishing"
            ):
                renew_publish_lease(("Publishing",))
                reconcile_and_rollback_generation(
                    PUBLISH_GENERATION,
                    RUN_ID,
                    CURRENT_GENERATION_ID,
                    PREVIOUS_GENERATION_ID,
                    "Publishing",
                )
                rollback_completed = True
            elif (
                int(lease["Generation"]) == PUBLISH_GENERATION
                and lease["PublicationRunId"] == RUN_ID
                and lease["OwnerRunId"] == RUN_ID
                and lease["PublishState"] == "Preparing"
            ):
                rollback_completed = True
        except Exception as rollback_error:
            rollback_errors.append(str(rollback_error))
    failure_detail = (
        "Ready verification recovery CAS-rolled back every published table before release."
        if ready_failure_recovered
        else (
            "Reconciliation classified every intent and CAS-rolled back every committed table."
            if rollback_completed
            else f"Reconciliation refused release; exclusive ownership/evidence is retained: {rollback_errors}"
        )
    )
    failed_recorded = ready_failure_recovered
    if PUBLISH_GENERATION is not None and rollback_completed and not failed_recorded:
        try:
            failed_recorded = fail_owned_generation(
                f"Publication failed. {failure_detail} Cause: {publish_error}"
            )
        except Exception as state_error:
            rollback_errors.append(f"control-state: {state_error}")
    evidence_count = (
        generation_rows(PUBLISH_GENERATION)
        .where(F.col("RecordType") == "Table")
        .count()
        if PUBLISH_GENERATION is not None
        else 0
    )
    if evidence_count == 0:
        cleanup_errors = drop_tables_best_effort(
            [TEMP_TABLE_MAP[name] for name in FINAL_TABLE_ORDER]
        )
        if cleanup_errors:
            rollback_errors.extend(cleanup_errors)
    retained = [TEMP_TABLE_MAP[name] for name in FINAL_TABLE_ORDER] if evidence_count else []
    raise RuntimeError(
        "Atomic-CAS publication failed and is not Ready. "
        f"FailedStateRecorded={failed_recorded}. {failure_detail} "
        f"ValidatedTemporaryTablesRetained={retained}. "
        "Retained temporary tables are kept as evidence and are never reused by a "
        "later run; drop them with DROP TABLE once the failure has been reviewed. "
        "Cross-table multi-ACID atomicity is not claimed."
    ) from publish_error

if publish_succeeded:
    cleanup_errors = drop_tables_best_effort(
        [TEMP_TABLE_MAP[name] for name in FINAL_TABLE_ORDER]
    )
    if cleanup_errors:
        print(
            "Ready generation published; temporary cleanup requires review:",
            cleanup_errors,
        )


In [ ]:
print(
    f"READY: Furusato ontology data v{NOTEBOOK_VERSION}; "
    f"participant={PARTICIPANT_ID}; generation={PUBLISH_GENERATION}; "
    f"8 staging + 11 ontology + 1 load audit table."
)
print(
    f"All 20 final tables passed atomic generation CAS and control evidence in "
    f"{PUBLISH_CONTROL_TABLE}."
)
print(
    "Consumers must recheck Ready evidence before use. Per-table replacement is atomic, "
    "but cross-table multi-ACID atomicity is not claimed."
)
